This cell installs the necessary Python libraries for PDF processing, embedding, and vector database functionality. After identifying that `sentence-transformers` is also a required library, I've added it to the installation list to resolve the `ValueError` encountered during ChromaDB collection building.

In [1]:
import sys
if 'google.colab' in sys.modules:
  # Uninstall to clear potential conflicts
  !pip uninstall -y sentence-transformers transformers Pillow
  # Reinstall with specific order to resolve dependencies
  !pip install -q openai pdfplumber pypdf pdf2image pytesseract chromadb Pillow transformers sentence-transformers

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0


####PHASE 1
This cell defines data structures using `dataclasses` to hold information about page diagnostics and the overall PDF extraction report. These structures help organize the extracted data in a standardized format.

In [2]:
import argparse
import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pdfplumber


@dataclass
class PageDiagnostics:
	page_number: int
	char_count: int
	word_count: int
	table_count: int
	text_density: float
	likely_scanned: bool
	extraction_warnings: list[str]


@dataclass
class ExtractionReport:
	pdf_path: str
	total_pages: int
	readable_pages: int
	scanned_pages: int
	table_heavy_pages: list[int]
	section_heading_candidates: list[dict[str, Any]]
	ocr_recommended: bool
	generated_at: str


This function `_extract_heading_candidates` attempts to identify potential section headings within a page's text using regular expressions. It looks for numbered headings, all-caps lines, and title-cased lines to suggest logical sections.

In [3]:
def _extract_heading_candidates(page_text: str, page_number: int) -> list[dict[str, Any]]:
	# Heuristic heading detector used only for section-candidate discovery.
	candidates: list[dict[str, Any]] = []
	if not page_text:
		return candidates

	heading_patterns = [
		re.compile(r"^\d+(?:\.\d+)*\s+.+$"),
		re.compile(r"^[A-Z][A-Z\s&\-]{3,}$"),
	]

	for raw_line in page_text.splitlines():
		line = raw_line.strip()
		if not line:
			continue
		if len(line) > 120:
			continue

		numbered = bool(heading_patterns[0].match(line))
		all_caps = bool(heading_patterns[1].match(line))
		title_case = line == line.title() and len(line.split()) <= 14

		if numbered or all_caps or title_case:
			candidates.append(
				{
					"page_number": page_number,
					"heading": line,
					"kind": "numbered" if numbered else "all_caps" if all_caps else "title_case",
				}
			)

	return candidates


The `_safe_extract_page` function is designed to robustly extract text and tables from a given PDF page using `pdfplumber`. It handles potential errors during extraction by catching exceptions and recording warnings, ensuring that the process continues even if parts of a page cannot be fully extracted.

In [4]:
def _safe_extract_page(page: pdfplumber.page.Page) -> tuple[str, list[list[list[str | None]]], list[str]]:
	# Parse text and tables independently so one failure does not drop the page.
	warnings: list[str] = []

	text = ""
	try:
		text = page.extract_text() or ""
	except Exception as exc:  # noqa: BLE001
		warnings.append(f"text_extraction_error: {exc}")

	tables: list[list[list[str | None]]] = []
	try:
		extracted_tables = page.extract_tables() or []
		tables = [table for table in extracted_tables if table]
	except Exception as exc:  # noqa: BLE001
		warnings.append(f"table_extraction_error: {exc}")

	return text, tables, warnings

The `run_phase1_extraction` function orchestrates the initial extraction of content from a PDF. It opens the PDF, iterates through its pages, extracts text and tables, and collects diagnostics for each page. It then compiles an `ExtractionReport` and saves the extracted page data and report to JSON files. This phase identifies potentially scanned pages and large tables.

In [5]:
def run_phase1_extraction(
	pdf_path: Path,
	out_dir: Path,
	min_chars_threshold: int,
	table_heavy_threshold: int,
	ocr_ratio_threshold: float,
) -> tuple[ExtractionReport, Path, Path]:
	# Validate input early to fail fast with a clear message.
	if not pdf_path.exists():
		raise FileNotFoundError(f"PDF path does not exist: {pdf_path}")

	out_dir.mkdir(parents=True, exist_ok=True)
	report_path = out_dir / "phase1_extraction_report.json"
	pages_path = out_dir / "phase1_pages.jsonl"

	diagnostics: list[PageDiagnostics] = []
	heading_candidates: list[dict[str, Any]] = []
	table_heavy_pages: list[int] = []

	with pdfplumber.open(pdf_path) as pdf, pages_path.open("w", encoding="utf-8") as pages_file:
		# Total pages is useful for quick console diagnostics.
		total_pages = len(pdf.pages)

		for idx, page in enumerate(pdf.pages, start=1):
			# Page-level quality signals drive OCR recommendation later.
			page_text, tables, warnings = _safe_extract_page(page)
			clean_text = page_text.strip()
			word_count = len(clean_text.split()) if clean_text else 0
			char_count = len(clean_text)
			text_density = round(word_count / max(char_count, 1), 6)

			likely_scanned = char_count < min_chars_threshold
			table_count = len(tables)

			if table_count >= table_heavy_threshold:
				table_heavy_pages.append(idx)

			heading_candidates.extend(_extract_heading_candidates(clean_text, idx))

			page_diag = PageDiagnostics(
				page_number=idx,
				char_count=char_count,
				word_count=word_count,
				table_count=table_count,
				text_density=text_density,
				likely_scanned=likely_scanned,
				extraction_warnings=warnings,
			)
			diagnostics.append(page_diag)

			page_record: dict[str, Any] = {
				"page_number": idx,
				"text": clean_text,
				"tables": tables,
				"diagnostics": asdict(page_diag),
			}
			pages_file.write(json.dumps(page_record, ensure_ascii=False) + "\n")

	# OCR recommendation is based on scanned-page ratio threshold.
	scanned_pages = sum(1 for d in diagnostics if d.likely_scanned)
	readable_pages = len(diagnostics) - scanned_pages
	ocr_recommended = (scanned_pages / max(len(diagnostics), 1)) >= ocr_ratio_threshold

	report = ExtractionReport(
		pdf_path=str(pdf_path),
		total_pages=len(diagnostics),
		readable_pages=readable_pages,
		scanned_pages=scanned_pages,
		table_heavy_pages=table_heavy_pages,
		section_heading_candidates=heading_candidates,
		ocr_recommended=ocr_recommended,
		generated_at=datetime.now(timezone.utc).isoformat(),
	)

	with report_path.open("w", encoding="utf-8") as report_file:
		json.dump(asdict(report), report_file, indent=2, ensure_ascii=False)

	return report, report_path, pages_path


This markdown cell provides instructions for the user to upload a PDF file. Once uploaded, the file's path will be stored in the `pdf_path` variable, which is crucial for subsequent processing steps.

Please upload the PDF file you wish to process. Once uploaded, the file path will be stored in the `pdf_path` variable.

This cell handles the file upload process in Google Colab. It uses `files.upload()` to allow the user to select a PDF. Upon successful upload, it stores the path of the uploaded PDF in the `pdf_path` variable.

In [6]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        pdf_path = Path(filename)
        print(f'Uploaded file: {pdf_path}')
        break # Assuming only one PDF is uploaded
else:
    print('No file uploaded.')


Saving XYZ Analytics Consulting – Product & Solutions Handbook.pdf to XYZ Analytics Consulting – Product & Solutions Handbook (1).pdf
Uploaded file: XYZ Analytics Consulting – Product & Solutions Handbook (1).pdf


This cell defines `out_dir` as the directory where all output files (extraction reports, structured entities, chunks) will be stored. It ensures the directory exists, creating it if necessary.

In [7]:
# Define the output directory to store the extraction report and page data
out_dir = Path("extraction_output")
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory created at: {out_dir}")

Output directory created at: extraction_output


This cell initializes an `Args` class to hold configuration parameters for the PDF extraction process. These parameters, such as `min_chars_threshold`, `table_heavy_threshold`, and `ocr_ratio_threshold`, are used to tune the heuristic analysis of PDF content quality.

In [8]:
min_chars = 100  # Minimum characters for a page to be considered readable
table_heavy = 3  # Minimum number of tables for a page to be considered 'table heavy'
ocr_ratio = 0.5  # If more than 50% of pages are likely scanned, OCR is recommended


This cell executes the `run_phase1_extraction` function, which performs the initial processing of the uploaded PDF. It uses the `pdf_path` and the defined `args` to extract text, tables, and generate a diagnostic report, saving the results to the specified output directory.

In [9]:
report, report_path, pages_path = run_phase1_extraction(
		pdf_path=pdf_path,
		out_dir=out_dir,
		min_chars_threshold = min_chars,
		table_heavy_threshold = table_heavy,
		ocr_ratio_threshold = ocr_ratio,
	)

###Verify Phase 1 - file generated under extraction_output folder.

Sample data:
{
"pdf_path": "XYZ Analytics Consulting – Product & Solutions Handbook (1).pdf",
  "total_pages": 20,
  "readable_pages": 20,
  "scanned_pages": 0,
  "table_heavy_pages": [
    13,
    15,
    16
  ],
  "section_heading_candidates": [
    {
      "page_number": 1,
      "heading": "Automotive Analytics Solutions Confidential",
      "kind": "title_case"
    }
  }

####PHASE 2
This cell defines data structures and regular expressions crucial for Phase 2, which focuses on extracting specific entities like services, pricing, timelines, etc., from the cleaned PDF text. It also defines keywords and patterns to identify these entities within the document.

In [10]:
import argparse
import json
import re
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

@dataclass
class EntityRecord:
    value: str
    page_number: int
    source_type: str
    evidence: str
    confidence: float


@dataclass
class Phase2Output:
    generated_at: str
    source_pages_file: str
    source_report_file: str | None
    entities: dict[str, list[EntityRecord]] = field(default_factory=dict)
    stats: dict[str, Any] = field(default_factory=dict)


SERVICE_HEADING_PATTERN = re.compile(r"^\s*\d+\.\s+(.+?)\s*$")
# Timeline pattern catches both "Months 1-3" and generic phase markers.
TIMELINE_PATTERN = re.compile(r"\b(?:Months?\s*\d+\s*[\-\u2013]\s*\d+|\d+\s*[\-\u2013]\s*\d+\s*weeks?|Phase\s*\d+)\b", re.IGNORECASE)
CURRENCY_PATTERN = re.compile(r"(?:₹|INR|USD|\$)\s?\d[\d,]*(?:\.\d+)?", re.IGNORECASE)

PRICING_KEYWORDS = (
    "pricing",
    "quoted",
    "fee",
    "fees",
    "retainer",
    "lump-sum",
    "commercial",
    "engagement model",
    "subscription",
    "lakhs",
    "crores",
)

SERVICE_NAME_HINTS = (
    "analytics",
    "prediction",
    "intelligence",
)

NON_PRICING_CONTEXT = (
    "gdp",
    "global auto sales",
    "market",
)

APPROACH_KEYWORDS = (
    "technical approach",
    "data ingestion",
    "pipeline",
    "etl",
    "elt",
    "machine learning",
    "model",
    "visualisation",
    "alerts",
    "security",
)

SLA_KEYWORDS = (
    "sla",
    "support",
    "response",
    "availability",
    "monitoring",
)

EXCLUSION_KEYWORDS = (
    "out of scope",
    "not included",
    "does not include",
    "excluding",
    "not covered",
)

ASSUMPTION_KEYWORDS = (
    "minimum",
    "requires",
    "assumes",
    "assuming",
    "depends",
    "if available",
    "nda",
)

ENGAGEMENT_MODEL_PATTERNS = (
    (re.compile(r"fixed\s*-?\s*price\s+project", re.IGNORECASE), "Fixed-Price Project"),
    (re.compile(r"time\s*&\s*materials", re.IGNORECASE), "Time & Materials"),
    (re.compile(r"managed\s+service\s*/\s*subscription", re.IGNORECASE), "Managed Service / Subscription"),
    (re.compile(r"revenue\s*-?\s*share\s*/\s*success(?:\s*fee)?", re.IGNORECASE), "Revenue-Share / Success Fee"),
)

This collection of helper functions supports the entity extraction process in Phase 2. Functions like `_normalize_text`, `_split_sentences`, and `_flatten_table_rows` preprocess the text. `_add_entity` handles the storage and deduplication of extracted entities, while `extract_phase2_entities` applies rule-based logic to identify and categorize various information from the PDF pages.

In [11]:
def _normalize_text(text: str) -> str:
    # Normalize punctuation and spacing so regex and dedupe stay stable.
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


def _split_sentences(text: str) -> list[str]:
    chunks = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [_normalize_text(chunk) for chunk in chunks if chunk and _normalize_text(chunk)]


def _is_toc_line(line: str) -> bool:
    return bool(re.search(r"\.{4,}\s*\d+\s*$", line))


def _load_phase1_pages(pages_file: Path) -> list[dict[str, Any]]:
    if not pages_file.exists():
        raise FileNotFoundError(f"Phase 1 pages file not found: {pages_file}")

    pages: list[dict[str, Any]] = []
    with pages_file.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                pages.append(json.loads(line))
    return pages


def _flatten_table_rows(tables: list[Any]) -> list[str]:
    # Flatten PDF table cells into readable row strings for rule-based parsing.
    rows: list[str] = []
    for table in tables or []:
        for row in table or []:
            values = [str(cell).strip() for cell in row if cell is not None and str(cell).strip()]
            if values:
                rows.append(_normalize_text(" | ".join(values)))
    return rows


def _contains_keyword(text: str, keywords: tuple[str, ...]) -> bool:
    lowered = text.lower()
    return any(keyword in lowered for keyword in keywords)


def _add_entity(
    bucket: list[EntityRecord],
    dedupe: set[tuple[str, int]],
    value: str,
    page_number: int,
    source_type: str,
    evidence: str,
    confidence: float,
) -> None:
    # Deduping by value+page avoids repeated fragments from broken PDF layout.
    clean_value = _normalize_text(value)
    clean_evidence = _normalize_text(evidence)
    key = (clean_value.lower(), page_number)

    if not clean_value or key in dedupe:
        return

    dedupe.add(key)
    bucket.append(
        EntityRecord(
            value=clean_value,
            page_number=page_number,
            source_type=source_type,
            evidence=clean_evidence,
            confidence=round(confidence, 2),
        )
    )


def extract_phase2_entities(
    pages: list[dict[str, Any]],
) -> dict[str, list[EntityRecord]]:
    # Entity buckets are intentionally explicit to support strict filtering.
    entities: dict[str, list[EntityRecord]] = {
        "services": [],
        "pricing": [],
        "delivery_approach": [],
        "timelines": [],
        "sla_terms": [],
        "exclusions": [],
        "assumptions": [],
        "engagement_models": [],
    }

    dedupe_map: dict[str, set[tuple[str, int]]] = defaultdict(set)

    for page in pages:
        # Page-level context helps avoid misclassifying TOC and unrelated text.
        page_number = int(page.get("page_number", 0))
        text = str(page.get("text", ""))
        lowered_text = text.lower()
        is_pricing_page = "pricing & engagement models" in lowered_text or "phased commercial" in lowered_text
        is_service_page = "service descriptions" in lowered_text or (
            "what we do" in lowered_text and "key capabilities" in lowered_text
        )
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        sentences = _split_sentences(text)
        table_rows = _flatten_table_rows(page.get("tables", []))

        for line in lines:
            if _is_toc_line(line):
                continue
            if not is_service_page:
                continue
            service_match = SERVICE_HEADING_PATTERN.match(line)
            if service_match:
                service_name = service_match.group(1).strip()
                if not any(hint in service_name.lower() for hint in SERVICE_NAME_HINTS):
                    continue
                _add_entity(
                    entities["services"],
                    dedupe_map["services"],
                    service_name,
                    page_number,
                    "text",
                    line,
                    0.95,
                )

        for sentence in sentences:
            if _is_toc_line(sentence):
                continue
            # Pricing extraction is conservative to reduce false positives.
            sentence_lower = sentence.lower()
            monetary_signal = bool(CURRENCY_PATTERN.search(sentence) or re.search(r"\b(?:lakhs?|crores?)\b", sentence_lower))
            pricing_signal = _contains_keyword(sentence, PRICING_KEYWORDS)
            noisy_pricing_context = any(token in sentence_lower for token in NON_PRICING_CONTEXT)

            if (is_pricing_page and pricing_signal) or (monetary_signal and pricing_signal and not noisy_pricing_context):
                _add_entity(
                    entities["pricing"],
                    dedupe_map["pricing"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.8,
                )

            if _contains_keyword(sentence, APPROACH_KEYWORDS):
                _add_entity(
                    entities["delivery_approach"],
                    dedupe_map["delivery_approach"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.78,
                )

            if TIMELINE_PATTERN.search(sentence):
                _add_entity(
                    entities["timelines"],
                    dedupe_map["timelines"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.82,
                )

            if _contains_keyword(sentence, SLA_KEYWORDS):
                _add_entity(
                    entities["sla_terms"],
                    dedupe_map["sla_terms"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.74,
                )

            if _contains_keyword(sentence, EXCLUSION_KEYWORDS):
                _add_entity(
                    entities["exclusions"],
                    dedupe_map["exclusions"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.73,
                )

            if _contains_keyword(sentence, ASSUMPTION_KEYWORDS):
                _add_entity(
                    entities["assumptions"],
                    dedupe_map["assumptions"],
                    sentence,
                    page_number,
                    "text",
                    sentence,
                    0.72,
                )

        for row in table_rows:
            # Engagement models are often split across rows/cells in PDF tables.
            row_lower = row.lower()

            for pattern, canonical_model in ENGAGEMENT_MODEL_PATTERNS:
                if pattern.search(row):
                    _add_entity(
                        entities["engagement_models"],
                        dedupe_map["engagement_models"],
                        canonical_model,
                        page_number,
                        "table",
                        row,
                        0.92,
                    )

            for pattern, canonical_model in ENGAGEMENT_MODEL_PATTERNS:
                if pattern.search(text):
                    _add_entity(
                        entities["engagement_models"],
                        dedupe_map["engagement_models"],
                        canonical_model,
                        page_number,
                        "text",
                        canonical_model,
                        0.9,
                    )

            if is_pricing_page and (_contains_keyword(row, PRICING_KEYWORDS) or CURRENCY_PATTERN.search(row)):
                _add_entity(
                    entities["pricing"],
                    dedupe_map["pricing"],
                    row,
                    page_number,
                    "table",
                    row,
                    0.84,
                )

            if TIMELINE_PATTERN.search(row):
                _add_entity(
                    entities["timelines"],
                    dedupe_map["timelines"],
                    row,
                    page_number,
                    "table",
                    row,
                    0.88,
                )

            if _contains_keyword(row, SLA_KEYWORDS):
                _add_entity(
                    entities["sla_terms"],
                    dedupe_map["sla_terms"],
                    row,
                    page_number,
                    "table",
                    row,
                    0.76,
                )

    return entities


The `run_phase2` function orchestrates the second phase of the pipeline. It loads the pre-processed pages from Phase 1, extracts structured entities using the `extract_phase2_entities` function, and compiles these entities into a `Phase2Output` report. This report is then saved to a JSON file.

In [12]:
def run_phase2(
    phase1_pages_file: Path,
    output_file: Path,
    source_report_file: Path | None = None,
) -> Path:
    pages = _load_phase1_pages(phase1_pages_file)
    entities = extract_phase2_entities(pages)

    stats = {
        "total_pages": len(pages),
        "entity_counts": {k: len(v) for k, v in entities.items()},
        "pages_with_entities": sorted(
            {
                record.page_number
                for bucket in entities.values()
                for record in bucket
            }
        ),
    }

    payload = Phase2Output(
        generated_at=datetime.now(timezone.utc).isoformat(),
        source_pages_file=str(phase1_pages_file),
        source_report_file=str(source_report_file) if source_report_file else None,
        entities=entities,
        stats=stats,
    )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    serializable = asdict(payload)
    with output_file.open("w", encoding="utf-8") as handle:
        json.dump(serializable, handle, indent=2, ensure_ascii=False)

    return output_file

This cell executes the `run_phase2` function, using the `pages.jsonl` file (output from Phase 1) and generates a structured entities JSON file. This step extracts key information like pricing, services, and timelines from the document.

In [13]:
report_path = Path("/content/extraction_output/phase1_extraction_report.json")
output_file = run_phase2(
        phase1_pages_file=Path("/content/extraction_output/phase1_pages.jsonl"),
        source_report_file=report_path if report_path.exists() else None,
        output_file=Path("/content/extraction_output/phase2_structured_entities.json"),
    )

This cell defines the `ChunkRecord` dataclass and various helper functions that are part of Phase 3. Phase 3 focuses on breaking down the extracted content into smaller, meaningful chunks and assigning metadata to each chunk, such as section, subsection, and entity tags. This is crucial for efficient retrieval later on.

In [14]:
import argparse
import hashlib
import json
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


HEADER_NOISE = "automotive analytics solutions confidential"
# Section rules provide deterministic routing for metadata-first retrieval.
SECTION_RULES: list[tuple[str, tuple[str, ...]]] = [
    ("executive_summary", ("executive summary",)),
    ("company_overview", ("company overview", "market context")),
    ("service_descriptions", ("service descriptions", "what we do", "key capabilities")),
    ("data_requirements", ("data requirements", "data source")),
    ("technical_approach", ("technical approach", "data ingestion & lake", "analytics & machine learning")),
    ("implementation_roadmap", ("implementation roadmap", "phase 1", "phase 2", "phase 3")),
    ("pricing_and_engagement", ("pricing & engagement models", "engagement model", "custom-quoted")),
    ("case_study", ("avoids seven-figure", "saved in six months")),
    ("sample_outreach", ("sample outreach materials", "one-pager highlights")),
    ("faq", ("frequently asked questions",)),
    ("contact_next_steps", ("contact & next steps", "initial call", "discovery workshop")),
]

SUBSECTION_HINTS: list[str] = [
    "what we do",
    "key capabilities",
    "business value",
    "deliverables",
    "technical approach",
    "pricing & engagement models",
    "implementation roadmap",
    "frequently asked questions",
]


@dataclass
class ChunkRecord:
    chunk_id: str
    doc_id: str
    content: str
    content_type: str
    section: str
    subsection: str | None
    page_start: int
    page_end: int
    citation_anchor: str
    has_tables: bool
    table_count: int
    chunk_length_chars: int
    source_hash: str
    entity_tags: list[str]
    confidence: float

This set of helper functions is used in Phase 3 for loading JSON files, specifically the `pages.jsonl` from Phase 1 and the `structured_entities.json` from Phase 2. It also includes `_entity_map_by_page` to create a quick lookup for entity tags per page, which is used to enrich the metadata of the generated chunks.

In [15]:
def _normalize(text: str) -> str:
    # Fix common mojibake and normalize whitespace from PDF extraction artifacts.
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u00a0", " ")
    text = text.replace("â€¢", "- ")
    text = text.replace("â€”", "-")
    text = text.replace("â€“", "-")
    text = text.replace("â€”,", "-")
    text = text.replace("â€”,", "-")
    text = text.replace("â€", "")
    text = text.replace("â", "")
    return re.sub(r"\s+", " ", text).strip()


def _is_noise_line(line: str) -> bool:
    # Remove repetitive headers/footers/TOC trails before chunk construction.
    clean = _normalize(line).lower()
    if not clean:
        return True
    if clean == HEADER_NOISE:
        return True
    if "table of contents" in clean:
        return True
    if re.search(r"^©\s*\d{4}.*page\s*\d+", clean):
        return True
    if re.search(r"all rights reserved\.?(?:\s*page\s*\d+)?$", clean):
        return True
    if re.search(r"\.{4,}\s*\d+\s*$", clean):
        return True
    return False


def _is_low_value_content(text: str) -> bool:
    # Guard against tiny, low-information fragments becoming standalone chunks.
    clean = _normalize(text).lower()
    if not clean:
        return True
    if "table of contents" in clean:
        return True
    if len(clean) < 120 and re.search(r"^©\s*\d{4}|all rights reserved|^page\s*\d+", clean):
        return True
    alpha_count = sum(1 for c in clean if c.isalpha())
    return alpha_count < 40


def _split_sentences(text: str) -> list[str]:
    raw = re.split(r"(?<=[.!?])\s+", text)
    return [_normalize(x) for x in raw if _normalize(x)]


def _guess_section(page_text: str, fallback: str) -> str:
    # Keep prior section as fallback to preserve continuity across related pages.
    lowered = page_text.lower()
    for section, keywords in SECTION_RULES:
        if any(keyword in lowered for keyword in keywords):
            return section
    return fallback


def _guess_subsection(text: str) -> str | None:
    lowered = text.lower()
    for hint in SUBSECTION_HINTS:
        if hint in lowered:
            return hint.replace(" ", "_")
    return None


def _flatten_table(table: list[list[Any]]) -> str:
    rows: list[str] = []
    for row in table or []:
        vals = [str(cell).strip() for cell in row if cell is not None and str(cell).strip()]
        if vals:
            rows.append(" | ".join(vals))
    return _normalize("\n".join(rows))

These functions in Phase 3 are responsible for building the individual content chunks. `_build_chunk_id` generates unique IDs for chunks, and `_chunk_confidence` assigns a confidence score. The `build_chunks` function performs the core logic of segmenting the document into text and table chunks, incorporating section guessing and entity tags.

In [16]:
def _load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _load_pages_jsonl(path: Path) -> list[dict[str, Any]]:
    pages: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                pages.append(json.loads(line))
    return pages


def _entity_map_by_page(entities_doc: dict[str, Any]) -> dict[int, set[str]]:
    # Convert Phase 2 entities into quick lookup tags for chunk metadata.
    page_tags: dict[int, set[str]] = {}
    entities = entities_doc.get("entities", {})
    for entity_type, records in entities.items():
        for record in records:
            page = int(record.get("page_number", 0))
            if page <= 0:
                continue
            if page not in page_tags:
                page_tags[page] = set()
            page_tags[page].add(entity_type)
    return page_tags

The `write_phase3_artifacts` function is responsible for writing the generated chunks to a JSONL file and creating a summary report. This report provides an overview of the chunking process, including chunk counts, content types, and average chunk lengths.

In [17]:
def _build_chunk_id(doc_id: str, page_start: int, page_end: int, idx: int) -> str:
    return f"{doc_id}_p{page_start:03d}_{page_end:03d}_c{idx:04d}"


def _chunk_confidence(content_type: str, tags: list[str]) -> float:
    base = 0.92 if content_type == "table" else 0.82
    if tags:
        base += 0.06
    return min(round(base, 2), 0.98)


def build_chunks(
    pages: list[dict[str, Any]],
    entity_page_tags: dict[int, set[str]],
    doc_id: str,
    max_chars: int,
    overlap_chars: int,
) -> list[ChunkRecord]:
    # Hybrid strategy:
    # 1) sentence-assembled text chunks with overlap
    # 2) table-preserving chunks as separate records
    chunks: list[ChunkRecord] = []
    current_section = "general"
    chunk_index = 1

    for page in pages:
        page_number = int(page.get("page_number", 0))
        text = str(page.get("text", ""))
        lines = [line.strip() for line in text.splitlines() if not _is_noise_line(line)]
        cleaned_text = _normalize("\n".join(lines))
        if cleaned_text:
            current_section = _guess_section(cleaned_text, current_section)

            sentences = _split_sentences(cleaned_text)
            buffer = ""
            for sentence in sentences:
                candidate = f"{buffer} {sentence}".strip() if buffer else sentence
                if len(candidate) <= max_chars:
                    buffer = candidate
                    continue

                if buffer:
                    if _is_low_value_content(buffer):
                        tail = buffer[-overlap_chars:] if overlap_chars > 0 else ""
                        buffer = f"{tail} {sentence}".strip() if tail else sentence
                        continue
                    subsection = _guess_subsection(buffer)
                    source_hash = hashlib.sha256(buffer.encode("utf-8")).hexdigest()[:16]
                    tags = sorted(entity_page_tags.get(page_number, set()))
                    chunk = ChunkRecord(
                        chunk_id=_build_chunk_id(doc_id, page_number, page_number, chunk_index),
                        doc_id=doc_id,
                        content=buffer,
                        content_type="text",
                        section=current_section,
                        subsection=subsection,
                        page_start=page_number,
                        page_end=page_number,
                        citation_anchor=f"page:{page_number}",
                        has_tables=bool(page.get("tables")),
                        table_count=len(page.get("tables", [])),
                        chunk_length_chars=len(buffer),
                        source_hash=source_hash,
                        entity_tags=tags,
                        confidence=_chunk_confidence("text", tags),
                    )
                    chunks.append(chunk)
                    chunk_index += 1

                    tail = buffer[-overlap_chars:] if overlap_chars > 0 else ""
                    buffer = f"{tail} {sentence}".strip()
                else:
                    buffer = sentence

            if buffer:
                if _is_low_value_content(buffer):
                    buffer = ""
                if not buffer:
                    continue
                subsection = _guess_subsection(buffer)
                source_hash = hashlib.sha256(buffer.encode("utf-8")).hexdigest()[:16]
                tags = sorted(entity_page_tags.get(page_number, set()))
                chunk = ChunkRecord(
                    chunk_id=_build_chunk_id(doc_id, page_number, page_number, chunk_index),
                    doc_id=doc_id,
                    content=buffer,
                    content_type="text",
                    section=current_section,
                    subsection=subsection,
                    page_start=page_number,
                    page_end=page_number,
                    citation_anchor=f"page:{page_number}",
                    has_tables=bool(page.get("tables")),
                    table_count=len(page.get("tables", [])),
                    chunk_length_chars=len(buffer),
                    source_hash=source_hash,
                    entity_tags=tags,
                    confidence=_chunk_confidence("text", tags),
                )
                chunks.append(chunk)
                chunk_index += 1

        for table in page.get("tables", []):
            # Tables are stored intact to avoid losing row/column semantics.
            table_text = _flatten_table(table)
            if not table_text or _is_low_value_content(table_text):
                continue
            source_hash = hashlib.sha256(table_text.encode("utf-8")).hexdigest()[:16]
            tags = sorted(entity_page_tags.get(page_number, set()))
            table_chunk = ChunkRecord(
                chunk_id=_build_chunk_id(doc_id, page_number, page_number, chunk_index),
                doc_id=doc_id,
                content=table_text,
                content_type="table",
                section=current_section,
                subsection="table",
                page_start=page_number,
                page_end=page_number,
                citation_anchor=f"page:{page_number}:table",
                has_tables=True,
                table_count=1,
                chunk_length_chars=len(table_text),
                source_hash=source_hash,
                entity_tags=tags,
                confidence=_chunk_confidence("table", tags),
            )
            chunks.append(table_chunk)
            chunk_index += 1

    return chunks

The `run_phase3` function orchestrates the chunking process. It loads the pages and entities, builds the chunks using the specified parameters (`max_chars`, `overlap_chars`), and then writes these chunks and a summary report to the output directory.

In [18]:
def write_phase3_artifacts(
    chunks: list[ChunkRecord],
    output_dir: Path,
    source_pages_file: Path,
    source_entities_file: Path,
) -> tuple[Path, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    chunks_path = output_dir / "chunks.jsonl"
    summary_path = output_dir / "chunk_report.json"

    with chunks_path.open("w", encoding="utf-8") as handle:
        for chunk in chunks:
            handle.write(json.dumps(chunk.__dict__, ensure_ascii=False) + "\n")

    section_counts: dict[str, int] = {}
    content_type_counts: dict[str, int] = {}
    for chunk in chunks:
        section_counts[chunk.section] = section_counts.get(chunk.section, 0) + 1
        content_type_counts[chunk.content_type] = content_type_counts.get(chunk.content_type, 0) + 1

    report = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source_pages_file": str(source_pages_file),
        "source_entities_file": str(source_entities_file),
        "total_chunks": len(chunks),
        "text_chunks": content_type_counts.get("text", 0),
        "table_chunks": content_type_counts.get("table", 0),
        "sections": section_counts,
        "avg_chunk_chars": round(sum(c.chunk_length_chars for c in chunks) / max(len(chunks), 1), 2),
    }

    with summary_path.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)

    return chunks_path, summary_path

This cell executes the `run_phase3` function, generating chunks of the document based on the processed pages and extracted entities. The chunks are saved to a `chunks.jsonl` file within the output directory, along with a chunk report.

In [19]:
def run_phase3(
    phase1_pages_file: Path,
    phase2_entities_file: Path,
    out_dir: Path,
    doc_id: str,
    max_chars: int,
    overlap_chars: int,
) -> tuple[Path, Path]:
    pages = _load_pages_jsonl(phase1_pages_file)
    entities = _load_json(phase2_entities_file)
    page_tags = _entity_map_by_page(entities)

    chunks = build_chunks(
        pages=pages,
        entity_page_tags=page_tags,
        doc_id=doc_id,
        max_chars=max_chars,
        overlap_chars=overlap_chars,
    )

    return write_phase3_artifacts(
        chunks=chunks,
        output_dir=out_dir,
        source_pages_file=phase1_pages_file,
        source_entities_file=phase2_entities_file,
    )

This cell defines the `RetrievalHit` dataclass and helper functions for Phase 4, which involves indexing the generated chunks in ChromaDB and performing metadata-first retrieval. It sets up the structure for storing and retrieving relevant chunks based on a query.

In [20]:
chunks_path, summary_path = run_phase3(
        phase1_pages_file=Path("/content/extraction_output/phase1_pages.jsonl"),
        phase2_entities_file=Path("/content/extraction_output/phase2_structured_entities.json"),
        out_dir=Path("/content/extraction_output"),
        doc_id="xyz_handbook_v1",
        max_chars=1800,
        overlap_chars=250,
    )

#### PHASE 4
This cell defines the `RetrievalHit` dataclass and various helper functions that are part of Phase 4. Phase 4 focuses on indexing the generated chunks in ChromaDB and performing metadata-first retrieval. It sets up the structure for storing and retrieving relevant chunks based on a query.

In [21]:
"""Phase 4 pipeline: index chunks in Chroma and run strict metadata-first retrieval."""

import argparse
import json
import os
import re
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import chromadb
from chromadb.utils import embedding_functions


@dataclass
class RetrievalHit:
    chunk_id: str
    score: float
    page_start: int
    page_end: int
    section: str
    citation_anchor: str
    content_preview: str
    confidence: float
    entity_tags: list[str]

This section contains helper functions for loading chunks, sanitizing metadata for ChromaDB, and building the Chroma collection. `_load_chunks` reads the chunk files, `_sanitize_metadata` prepares chunk metadata for indexing, and `_build_collection` sets up the ChromaDB collection with a Sentence Transformer embedding function.

In [22]:

def _load_chunks(chunks_file: Path) -> list[dict[str, Any]]:
    if not chunks_file.exists():
        raise FileNotFoundError(f"Chunks file not found: {chunks_file}")

    chunks: list[dict[str, Any]] = []
    with chunks_file.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                chunks.append(json.loads(line))
    return chunks


def _sanitize_metadata(chunk: dict[str, Any], run_id: str) -> dict[str, Any]:
    # Normalize metadata to simple scalar values accepted by Chroma.
    tags = chunk.get("entity_tags", [])
    joined_tags = ",".join(sorted(tags)) if tags else ""

    return {
        "doc_id": str(chunk.get("doc_id", "")),
        "chunk_id": str(chunk.get("chunk_id", "")),
        "content_type": str(chunk.get("content_type", "text")),
        "section": str(chunk.get("section", "general")),
        "subsection": str(chunk.get("subsection") or ""),
        "page_start": int(chunk.get("page_start", 0)),
        "page_end": int(chunk.get("page_end", 0)),
        "citation_anchor": str(chunk.get("citation_anchor", "")),
        "has_tables": bool(chunk.get("has_tables", False)),
        "table_count": int(chunk.get("table_count", 0)),
        "chunk_length_chars": int(chunk.get("chunk_length_chars", 0)),
        "source_hash": str(chunk.get("source_hash", "")),
        "confidence": float(chunk.get("confidence", 0.0)),
        "entity_tags": joined_tags,
        "run_id": run_id,
    }


def _build_collection(client: chromadb.ClientAPI, collection_name: str) -> chromadb.Collection:
    # Local sentence-transformer keeps retrieval self-contained/offline-friendly.
    embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

    return client.get_or_create_collection(
        name=collection_name,
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"},
    )


The `index_chunks` function handles the indexing of chunks into ChromaDB. It loads the chunks, generates a unique `run_id`, and then adds the chunks to the Chroma collection in batches. It also includes an `_upsert_batch` helper for managing additions and updates to the collection, and `_sha256_file` for file integrity checks.

In [23]:

def index_chunks(
    chunks_file: Path,
    persist_dir: Path,
    collection_name: str,
    batch_size: int = 32,
) -> dict[str, Any]:
    # Each indexing run gets a run_id + source hash for traceability.
    persist_dir.mkdir(parents=True, exist_ok=True)
    chunks = _load_chunks(chunks_file)

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    source_file_hash = _sha256_file(chunks_file)

    client = chromadb.PersistentClient(path=str(persist_dir))
    collection = _build_collection(client, collection_name)

    ids: list[str] = []
    docs: list[str] = []
    metas: list[dict[str, Any]] = []

    indexed_count = 0
    for chunk in chunks:
        chunk_id = str(chunk.get("chunk_id"))
        content = str(chunk.get("content", "")).strip()
        if not chunk_id or not content:
            continue

        ids.append(chunk_id)
        docs.append(content)
        metas.append(_sanitize_metadata(chunk, run_id=run_id))

        if len(ids) >= batch_size:
            _upsert_batch(collection, ids, docs, metas)
            indexed_count += len(ids)
            ids, docs, metas = [], [], []

    if ids:
        _upsert_batch(collection, ids, docs, metas)
        indexed_count += len(ids)

    report = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "run_id": run_id,
        "collection_name": collection_name,
        "persist_dir": str(persist_dir),
        "source_chunks_file": str(chunks_file),
        "source_file_hash": source_file_hash,
        "indexed_chunks": indexed_count,
        "total_in_collection": collection.count(),
    }
    return report


def _upsert_batch(
    collection: chromadb.Collection,
    ids: list[str],
    docs: list[str],
    metas: list[dict[str, Any]],
) -> None:
    # Delete existing ids before add to keep reruns deterministic.
    existing = collection.get(ids=ids)
    if existing and existing.get("ids"):
        collection.delete(ids=ids)

    collection.add(
        ids=ids,
        documents=docs,
        metadatas=metas,
    )


def _sha256_file(path: Path) -> str:
    import hashlib

    h = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(8192)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


These functions are crucial for querying the ChromaDB in Phase 4. `_parse_entity_filter` categorizes user queries to enable metadata-first filtering. `_contains_entity_tag` checks for specific entity tags. The `metadata_first_query` function performs the actual retrieval, applying strict metadata and confidence filters to return relevant chunks. `_preview` generates a short snippet of content.

In [24]:

def _parse_entity_filter(query: str) -> tuple[str | None, str | None]:
    # Rule-based intent mapper used by strict mode and evaluator expectations.
    lowered = query.lower()
    if any(term in lowered for term in ("price", "pricing", "cost", "charge", "charges", "fee", "fees", "commercial", "proof of concept", "enterprise deployment", "enterprise deployments")):
        return "pricing", "pricing_and_engagement"
    if any(term in lowered for term in ("service", "offering", "capabilities", "solution")):
        return "services", "service_descriptions"
    if any(term in lowered for term in ("approach", "implementation", "architecture", "technical")):
        return "delivery_approach", "technical_approach"
    if any(term in lowered for term in ("sla", "support", "availability", "response")):
        return "sla_terms", None
    return None, None


def _contains_entity_tag(entity_tags_joined: str, tag: str) -> bool:
    if not entity_tags_joined:
        return False
    parts = [t.strip() for t in entity_tags_joined.split(",") if t.strip()]
    return tag in parts


def metadata_first_query(
    persist_dir: Path,
    collection_name: str,
    query_text: str,
    top_k: int,
    min_confidence: float,
    strict_mode: bool,
) -> dict[str, Any]:
    # Strict mode requires detectable intent; otherwise abstain immediately.
    client = chromadb.PersistentClient(path=str(persist_dir))
    collection = _build_collection(client, collection_name)

    entity_filter, section_hint = _parse_entity_filter(query_text)

    if strict_mode and entity_filter is None and section_hint is None:
        return {
            "query": query_text,
            "strict_mode": strict_mode,
            "entity_filter": entity_filter,
            "section_hint": section_hint,
            "min_confidence": min_confidence,
            "abstain": True,
            "abstain_reason": "Unsupported or ambiguous query intent for strict metadata retrieval.",
            "hits": [],
            "generated_at": datetime.now(timezone.utc).isoformat(),
        }

    # Pull a larger pool, then apply strict metadata/confidence filtering in code.
    raw = collection.query(
        query_texts=[query_text],
        n_results=max(top_k * 6, 24),
        include=["documents", "metadatas", "distances"],
    )

    metadatas = raw.get("metadatas", [[]])[0]
    documents = raw.get("documents", [[]])[0]
    distances = raw.get("distances", [[]])[0]

    hits: list[RetrievalHit] = []
    for idx, meta in enumerate(metadatas):
        if not meta:
            continue
        confidence = float(meta.get("confidence", 0.0))
        if confidence < min_confidence:
            continue

        section = str(meta.get("section", ""))
        tags_joined = str(meta.get("entity_tags", ""))

        if entity_filter and not _contains_entity_tag(tags_joined, entity_filter):
            continue
        if section_hint and section != section_hint and strict_mode:
            continue

        distance = float(distances[idx]) if idx < len(distances) and distances[idx] is not None else 1.0
        score = max(0.0, 1.0 - distance)
        doc = documents[idx] if idx < len(documents) else ""

        hits.append(
            RetrievalHit(
                chunk_id=str(meta.get("chunk_id", "")),
                score=round(score, 4),
                page_start=int(meta.get("page_start", 0)),
                page_end=int(meta.get("page_end", 0)),
                section=section,
                citation_anchor=str(meta.get("citation_anchor", "")),
                content_preview=_preview(doc),
                confidence=round(confidence, 2),
                entity_tags=[t.strip() for t in tags_joined.split(",") if t.strip()],
            )
        )

    hits = sorted(hits, key=lambda h: h.score, reverse=True)[:top_k]

    abstain = False
    abstain_reason = ""
    if strict_mode and not hits:
        abstain = True
        abstain_reason = (
            "No chunk satisfied strict metadata filter and confidence threshold for this query intent."
        )

    return {
        "query": query_text,
        "strict_mode": strict_mode,
        "entity_filter": entity_filter,
        "section_hint": section_hint,
        "min_confidence": min_confidence,
        "abstain": abstain,
        "abstain_reason": abstain_reason,
        "hits": [h.__dict__ for h in hits],
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }


def _preview(text: str, max_len: int = 260) -> str:
    clean = re.sub(r"\s+", " ", text).strip()
    if len(clean) <= max_len:
        return clean
    return clean[: max_len - 3] + "..."

This cell executes the `index_chunks` function, which takes the previously generated `chunks.jsonl` file and indexes its content into a ChromaDB collection. It specifies the persistence directory for the database and the collection name. The output, `report`, summarizes the indexing operation.

In [25]:
report = index_chunks(
            chunks_file=Path("/content/extraction_output/chunks.jsonl"),
            persist_dir=Path("/content/extraction_output/chroma_db"),
            collection_name="xyz_handbook_chunks",
            batch_size=32,
        )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

This cell defines the path where the indexing report will be saved. It creates a `Path` object for the `index_report.json` file, which will contain details about the ChromaDB indexing process.

In [26]:
index_report_file = Path("/content/extraction_output/index_report.json")

This cell writes the `report` generated by the `index_chunks` function to a JSON file at the specified `index_report_file` path. This persists the details of the ChromaDB indexing operation for future reference.

In [27]:
report_file = Path(index_report_file)
report_file.parent.mkdir(parents=True, exist_ok=True)
with report_file.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

This cell sets up the parameters for performing a query against the indexed ChromaDB. It defines the `persist_dir` (where the ChromaDB is stored), the `collection_name` to query, and the `query_text` that the user is asking. It also specifies the `query_result_path` where the results will be saved.

In [28]:
persist_dir = Path("/content/extraction_output/chroma_db")
collection_name = "xyz_handbook_chunks"
query_text = "Which services do you provide and what capabilities are included?"
query_result_path = Path("/content/extraction_output/query_result.json")

This cell executes the `metadata_first_query` function to retrieve relevant chunks from ChromaDB based on the `query_text` and specified filters. It sets parameters like `top_k`, `min_confidence`, and `strict_mode` to control the retrieval process. The results are then saved to the `query_result_path`.

In [29]:
result = metadata_first_query(
        persist_dir=persist_dir,
        collection_name=collection_name,
        query_text=query_text,
        top_k=5,
        min_confidence=0.85,
        strict_mode="true",
    )
out_file = query_result_path
out_file.parent.mkdir(parents=True, exist_ok=True)
with out_file.open("w", encoding="utf-8") as handle:
    json.dump(result, handle, indent=2)

This code block defines the functions and logic for Phase 5 of the pipeline, which focuses on assembling citation-backed answers. It includes helper functions for loading JSON data, deduplicating lists, building citations, cleaning sentences, and extracting summaries for pricing, services, and delivery approaches. The core function `_compose_answer` generates a deterministic answer based on retrieved chunks and entity filters. It also includes `_synthesize_with_openai` for an optional AI-powered answer polishing, which uses the OpenAI API (if an API key is available). The main function `run_phase5` orchestrates the entire answer generation process.

In [30]:
"""Phase 5 pipeline: assemble strict, citation-backed answers.

Default mode is deterministic/template-based; optional OpenAI mode can polish
wording while still grounding on strict retrieval evidence.
"""

import argparse
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from openai import OpenAI


def _load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _dedupe_keep_order(values: list[str]) -> list[str]:
    # Preserve original order while removing repeats from noisy extraction.
    seen: set[str] = set()
    out: list[str] = []
    for value in values:
        key = value.strip().lower()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(value.strip())
    return out


def _build_citations(hits: list[dict[str, Any]]) -> list[dict[str, Any]]:
    # Keep citation schema compact and stable for downstream consumers/UI.
    citations: list[dict[str, Any]] = []
    for hit in hits:
        citations.append(
            {
                "chunk_id": hit.get("chunk_id"),
                "citation_anchor": hit.get("citation_anchor"),
                "page_start": hit.get("page_start"),
                "page_end": hit.get("page_end"),
                "section": hit.get("section"),
                "score": hit.get("score"),
                "confidence": hit.get("confidence"),
                "content_preview": hit.get("content_preview"),
            }
        )
    return citations


def _clean_sentence(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_pricing_summary(phase2: dict[str, Any]) -> dict[str, Any]:
    # Pull pricing-relevant lines for deterministic narrative construction.
    entities = phase2.get("entities", {})
    pricing_records = entities.get("pricing", [])
    model_records = entities.get("engagement_models", [])

    models = _dedupe_keep_order([str(item.get("value", "")).strip() for item in model_records])

    pricing_text = [str(item.get("value", "")).strip() for item in pricing_records]
    charge_lines: list[str] = []
    for line in pricing_text:
        lower = line.lower()
        if any(k in lower for k in ("custom-quoted", "lakhs", "crores", "lump-sum", "retainer", "daily", "weekly")):
            charge_lines.append(_clean_sentence(line))
    charge_lines = _dedupe_keep_order(charge_lines)

    return {
        "engagement_models": models,
        "charge_notes": charge_lines[:6],
    }


def _extract_services_summary(phase2: dict[str, Any], hits: list[dict[str, Any]]) -> dict[str, Any]:
    entities = phase2.get("entities", {})
    service_records = entities.get("services", [])
    services = _dedupe_keep_order([str(item.get("value", "")).strip() for item in service_records])

    capability_lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("key capabilities", "what we do", "dashboard", "predict", "forecast", "alerts")):
                capability_lines.append(clean)
    capability_lines = _dedupe_keep_order(capability_lines)

    return {
        "services": services,
        "capability_notes": capability_lines[:5],
    }


def _extract_delivery_summary(hits: list[dict[str, Any]]) -> list[str]:
    lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("data ingestion", "pipeline", "machine learning", "security", "dashboard", "alerts")):
                lines.append(clean)
    return _dedupe_keep_order(lines)[:6]


def _compose_answer(
    query_result: dict[str, Any],
    phase2: dict[str, Any],
) -> dict[str, Any]:
    # Compose by intent so strict mode behavior stays deterministic and testable.
    hits = query_result.get("hits", [])
    entity_filter = query_result.get("entity_filter")

    if query_result.get("abstain", False):
        return {
            "status": "abstain",
            "answer": "I cannot provide a reliable answer under strict mode for this query.",
            "reason": query_result.get("abstain_reason", "No qualifying evidence."),
            "citations": [],
            "structured_answer": {},
        }

    citations = _build_citations(hits)

    if entity_filter == "pricing":
        summary = _extract_pricing_summary(phase2)
        models = summary.get("engagement_models", [])
        charge_notes = summary.get("charge_notes", [])

        lines: list[str] = []
        if models:
            lines.append("Engagement models offered: " + ", ".join(models) + ".")
        if charge_notes:
            lines.append("Charges are described as follows: " + " ".join(charge_notes[:3]))
        lines.append("Final commercials are confirmed after discovery/scoping, so exact fixed price cards are not listed in the handbook.")

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from pricing-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "engagement_models": models,
                "charge_notes": charge_notes,
            },
        }

    if entity_filter == "services":
        summary = _extract_services_summary(phase2, hits)
        services = summary.get("services", [])
        capability_notes = summary.get("capability_notes", [])

        lines = []
        if services:
            lines.append("Services provided: " + ", ".join(services) + ".")
        if capability_notes:
            lines.append("Capabilities include: " + " ".join(capability_notes[:3]))

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from service-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "services": services,
                "capability_notes": capability_notes,
            },
        }

    if entity_filter == "delivery_approach":
        approach_notes = _extract_delivery_summary(hits)
        answer = "Technical approach highlights: " + " ".join(approach_notes) if approach_notes else "No clear delivery approach lines were found."
        return {
            "status": "answered" if approach_notes else "abstain",
            "answer": answer,
            "reason": "Answered from delivery-approach-tagged chunks." if approach_notes else "Insufficient approach evidence.",
            "citations": citations if approach_notes else [],
            "structured_answer": {"approach_notes": approach_notes},
        }

    if entity_filter == "sla_terms":
        sla_lines = []
        for hit in hits:
            preview = str(hit.get("content_preview", ""))
            for sentence in re.split(r"(?<=[.!?])\s+", preview):
                clean = _clean_sentence(sentence)
                lower = clean.lower()
                if any(k in lower for k in ("sla", "support", "monitoring", "response", "alerts")):
                    sla_lines.append(clean)
        sla_lines = _dedupe_keep_order(sla_lines)

        if not sla_lines:
            return {
                "status": "abstain",
                "answer": "I cannot provide a reliable SLA answer from available strict evidence.",
                "reason": "SLA intent matched, but no sufficiently explicit SLA lines were found.",
                "citations": [],
                "structured_answer": {},
            }

        return {
            "status": "answered",
            "answer": "SLA/support related points: " + " ".join(sla_lines[:4]),
            "reason": "Answered from SLA-tagged chunks.",
            "citations": citations,
            "structured_answer": {"sla_notes": sla_lines},
        }

    return {
        "status": "abstain",
        "answer": "I cannot answer this query under strict policy.",
        "reason": "Unsupported query intent for deterministic Phase 5 templates.",
        "citations": [],
        "structured_answer": {},
    }

def _synthesize_with_openai(
    query_text: str,
    deterministic_response: dict[str, Any],
    retrieval: dict[str, Any],
    model: str
) -> dict[str, Any]:
    # OpenAI synthesis is optional and never overrides strict abstain decisions.
    if deterministic_response.get("status") == "abstain":
        return {
            "status": "skipped",
            "answer": "",
            "reason": "Skipped OpenAI generation because strict-mode response abstained.",
            "model": model,
        }

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return {
            "status": "unavailable",
            "answer": "",
            "reason": "OPENAI_API_KEY was not found in environment or .env file.",
            "model": model,
        }

    client = OpenAI(api_key=api_key)

    citations = deterministic_response.get("citations", [])
    citation_lines = []
    for item in citations:
        anchor = item.get("citation_anchor", "")
        preview = item.get("content_preview", "")
        citation_lines.append(f"- {anchor}: {preview}")

    prompt = (
        # Prompt explicitly limits generation to retrieved evidence.
        "You are generating a strict, citation-grounded answer for a consulting handbook retrieval system. "
        "Use only the provided evidence snippets and do not invent details. "
        "If evidence does not provide exact numeric prices, explicitly state that pricing is custom-quoted. "
        "Return plain text, 4-8 sentences max, and include citation anchors inline like (page:16).\n\n"
        f"User query: {query_text}\n\n"
        f"Deterministic draft answer:\n{deterministic_response.get('answer', '')}\n\n"
        "Evidence snippets:\n"
        + "\n".join(citation_lines)
    )

    try:
        completion = client.responses.create(
            model=model,
            input=prompt,
            temperature=0,
        )
        text = completion.output_text.strip()
        return {
            "status": "answered",
            "answer": text,
            "reason": "Generated by OpenAI from strict retrieval evidence.",
            "model": model,
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "status": "error",
            "answer": "",
            "reason": f"OpenAI generation failed: {type(exc).__name__}",
            "model": model,
        }


def run_phase5(
    query_text: str,
    persist_dir: Path,
    collection_name: str,
    phase2_entities_file: Path,
    top_k: int,
    min_confidence: float,
    strict_mode: bool,
    use_openai: bool,
    openai_model: str
) -> dict[str, Any]:
    # Retrieval always runs first; answer generation is downstream of that result.
    query_result = metadata_first_query(
        persist_dir=persist_dir,
        collection_name=collection_name,
        query_text=query_text,
        top_k=top_k,
        min_confidence=min_confidence,
        strict_mode=strict_mode,
    )

    phase2 = _load_json(phase2_entities_file)
    answer_payload = _compose_answer(query_result, phase2)

    openai_payload: dict[str, Any] | None = None
    if use_openai:
        openai_payload = _synthesize_with_openai(
            query_text=query_text,
            deterministic_response=answer_payload,
            retrieval=query_result,
            model=openai_model
        )

    result = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "query": query_text,
        "strict_mode": strict_mode,
        "retrieval": query_result,
        "response": answer_payload,
    }
    if openai_payload is not None:
        result["openai_response"] = openai_payload
    return result

This cell is a duplicate of the previous Phase 5 pipeline definition (cell `jnx_a35nJdoN`). It redefines the same functions and logic for assembling citation-backed answers. It is likely a copy-paste error or an iterative development step that resulted in a duplicate.

In [31]:
"""Phase 5 pipeline: assemble strict, citation-backed answers.

Default mode is deterministic/template-based; optional OpenAI mode can polish
wording while still grounding on strict retrieval evidence.
"""

import argparse
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from openai import OpenAI


def _load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _dedupe_keep_order(values: list[str]) -> list[str]:
    # Preserve original order while removing repeats from noisy extraction.
    seen: set[str] = set()
    out: list[str] = []
    for value in values:
        key = value.strip().lower()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(value.strip())
    return out


def _build_citations(hits: list[dict[str, Any]]) -> list[dict[str, Any]]:
    # Keep citation schema compact and stable for downstream consumers/UI.
    citations: list[dict[str, Any]] = []
    for hit in hits:
        citations.append(
            {
                "chunk_id": hit.get("chunk_id"),
                "citation_anchor": hit.get("citation_anchor"),
                "page_start": hit.get("page_start"),
                "page_end": hit.get("page_end"),
                "section": hit.get("section"),
                "score": hit.get("score"),
                "confidence": hit.get("confidence"),
                "content_preview": hit.get("content_preview"),
            }
        )
    return citations


def _clean_sentence(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_pricing_summary(phase2: dict[str, Any]) -> dict[str, Any]:
    # Pull pricing-relevant lines for deterministic narrative construction.
    entities = phase2.get("entities", {})
    pricing_records = entities.get("pricing", [])
    model_records = entities.get("engagement_models", [])

    models = _dedupe_keep_order([str(item.get("value", "")).strip() for item in model_records])

    pricing_text = [str(item.get("value", "")).strip() for item in pricing_records]
    charge_lines: list[str] = []
    for line in pricing_text:
        lower = line.lower()
        if any(k in lower for k in ("custom-quoted", "lakhs", "crores", "lump-sum", "retainer", "daily", "weekly")):
            charge_lines.append(_clean_sentence(line))
    charge_lines = _dedupe_keep_order(charge_lines)

    return {
        "engagement_models": models,
        "charge_notes": charge_lines[:6],
    }


def _extract_services_summary(phase2: dict[str, Any], hits: list[dict[str, Any]]) -> dict[str, Any]:
    entities = phase2.get("entities", {})
    service_records = entities.get("services", [])
    services = _dedupe_keep_order([str(item.get("value", "")).strip() for item in service_records])

    capability_lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("key capabilities", "what we do", "dashboard", "predict", "forecast", "alerts")):
                capability_lines.append(clean)
    capability_lines = _dedupe_keep_order(capability_lines)

    return {
        "services": services,
        "capability_notes": capability_lines[:5],
    }


def _extract_delivery_summary(hits: list[dict[str, Any]]) -> list[str]:
    lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("data ingestion", "pipeline", "machine learning", "security", "dashboard", "alerts")):
                lines.append(clean)
    return _dedupe_keep_order(lines)[:6]


def _compose_answer(
    query_result: dict[str, Any],
    phase2: dict[str, Any],
) -> dict[str, Any]:
    # Compose by intent so strict mode behavior stays deterministic and testable.
    hits = query_result.get("hits", [])
    entity_filter = query_result.get("entity_filter")

    if query_result.get("abstain", False):
        return {
            "status": "abstain",
            "answer": "I cannot provide a reliable answer under strict mode for this query.",
            "reason": query_result.get("abstain_reason", "No qualifying evidence."),
            "citations": [],
            "structured_answer": {},
        }

    citations = _build_citations(hits)

    if entity_filter == "pricing":
        summary = _extract_pricing_summary(phase2)
        models = summary.get("engagement_models", [])
        charge_notes = summary.get("charge_notes", [])

        lines: list[str] = []
        if models:
            lines.append("Engagement models offered: " + ", ".join(models) + ".")
        if charge_notes:
            lines.append("Charges are described as follows: " + " ".join(charge_notes[:3]))
        lines.append("Final commercials are confirmed after discovery/scoping, so exact fixed price cards are not listed in the handbook.")

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from pricing-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "engagement_models": models,
                "charge_notes": charge_notes,
            },
        }

    if entity_filter == "services":
        summary = _extract_services_summary(phase2, hits)
        services = summary.get("services", [])
        capability_notes = summary.get("capability_notes", [])

        lines = []
        if services:
            lines.append("Services provided: " + ", ".join(services) + ".")
        if capability_notes:
            lines.append("Capabilities include: " + " ".join(capability_notes[:3]))

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from service-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "services": services,
                "capability_notes": capability_notes,
            },
        }

    if entity_filter == "delivery_approach":
        approach_notes = _extract_delivery_summary(hits)
        answer = "Technical approach highlights: " + " ".join(approach_notes) if approach_notes else "No clear delivery approach lines were found."
        return {
            "status": "answered" if approach_notes else "abstain",
            "answer": answer,
            "reason": "Answered from delivery-approach-tagged chunks." if approach_notes else "Insufficient approach evidence.",
            "citations": citations if approach_notes else [],
            "structured_answer": {"approach_notes": approach_notes},
        }

    if entity_filter == "sla_terms":
        sla_lines = []
        for hit in hits:
            preview = str(hit.get("content_preview", ""))
            for sentence in re.split(r"(?<=[.!?])\s+", preview):
                clean = _clean_sentence(sentence)
                lower = clean.lower()
                if any(k in lower for k in ("sla", "support", "monitoring", "response", "alerts")):
                    sla_lines.append(clean)
        sla_lines = _dedupe_keep_order(sla_lines)

        if not sla_lines:
            return {
                "status": "abstain",
                "answer": "I cannot provide a reliable SLA answer from available strict evidence.",
                "reason": "SLA intent matched, but no sufficiently explicit SLA lines were found.",
                "citations": [],
                "structured_answer": {},
            }

        return {
            "status": "answered",
            "answer": "SLA/support related points: " + " ".join(sla_lines[:4]),
            "reason": "Answered from SLA-tagged chunks.",
            "citations": citations,
            "structured_answer": {"sla_notes": sla_lines},
        }

    return {
        "status": "abstain",
        "answer": "I cannot answer this query under strict policy.",
        "reason": "Unsupported query intent for deterministic Phase 5 templates.",
        "citations": [],
        "structured_answer": {},
    }

def _synthesize_with_openai(
    query_text: str,
    deterministic_response: dict[str, Any],
    retrieval: dict[str, Any],
    model: str
) -> dict[str, Any]:
    # OpenAI synthesis is optional and never overrides strict abstain decisions.
    if deterministic_response.get("status") == "abstain":
        return {
            "status": "skipped",
            "answer": "",
            "reason": "Skipped OpenAI generation because strict-mode response abstained.",
            "model": model,
        }

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return {
            "status": "unavailable",
            "answer": "",
            "reason": "OPENAI_API_KEY was not found in environment or .env file.",
            "model": model,
        }

    client = OpenAI(api_key=api_key)

    citations = deterministic_response.get("citations", [])
    citation_lines = []
    for item in citations:
        anchor = item.get("citation_anchor", "")
        preview = item.get("content_preview", "")
        citation_lines.append(f"- {anchor}: {preview}")

    prompt = (
        # Prompt explicitly limits generation to retrieved evidence.
        "You are generating a strict, citation-grounded answer for a consulting handbook retrieval system. "
        "Use only the provided evidence snippets and do not invent details. "
        "If evidence does not provide exact numeric prices, explicitly state that pricing is custom-quoted. "
        "Return plain text, 4-8 sentences max, and include citation anchors inline like (page:16).\n\n"
        f"User query: {query_text}\n\n"
        f"Deterministic draft answer:\n{deterministic_response.get('answer', '')}\n\n"
        "Evidence snippets:\n"
        + "\n".join(citation_lines)
    )

    try:
        completion = client.responses.create(
            model=model,
            input=prompt,
            temperature=0,
        )
        text = completion.output_text.strip()
        return {
            "status": "answered",
            "answer": text,
            "reason": "Generated by OpenAI from strict retrieval evidence.",
            "model": model,
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "status": "error",
            "answer": "",
            "reason": f"OpenAI generation failed: {type(exc).__name__}",
            "model": model,
        }


def run_phase5(
    query_text: str,
    persist_dir: Path,
    collection_name: str,
    phase2_entities_file: Path,
    top_k: int,
    min_confidence: float,
    strict_mode: bool,
    use_openai: bool,
    openai_model: str
) -> dict[str, Any]:
    # Retrieval always runs first; answer generation is downstream of that result.
    query_result = metadata_first_query(
        persist_dir=persist_dir,
        collection_name=collection_name,
        query_text=query_text,
        top_k=top_k,
        min_confidence=min_confidence,
        strict_mode=strict_mode,
    )

    phase2 = _load_json(phase2_entities_file)
    answer_payload = _compose_answer(query_result, phase2)

    openai_payload: dict[str, Any] | None = None
    if use_openai:
        openai_payload = _synthesize_with_openai(
            query_text=query_text,
            deterministic_response=answer_payload,
            retrieval=query_result,
            model=openai_model
        )

    result = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "query": query_text,
        "strict_mode": strict_mode,
        "retrieval": query_result,
        "response": answer_payload,
    }
    if openai_payload is not None:
        result["openai_response"] = openai_payload
    return result

This cell is another duplicate of the Phase 5 pipeline definition. It redefines the same functions for answer generation, including the use of OpenAI for synthesis. It also imports `userdata` from `google.colab` to securely access API keys, which is an improvement over the previous versions that used `os.getenv`.

In [32]:
# Import the userdata module from google.colab
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")

This block defines the functions and logic for Phase 5 of the pipeline, which focuses on assembling citation-backed answers. It includes helper functions for loading JSON data, deduplicating lists, building citations, cleaning sentences, and extracting summaries for pricing, services, and delivery approaches. The core function `_compose_answer` generates a deterministic answer based on retrieved chunks and entity filters. It also includes `_synthesize_with_openai` for an optional AI-powered answer polishing, which uses the OpenAI API (if an API key is available). The main function `run_phase5` orchestrates the entire answer generation process.

This cell imports the `userdata` module from `google.colab` to securely retrieve the `OPENAI_API_KEY`.

In [33]:
"""Phase 5 pipeline: assemble strict, citation-backed answers.

Default mode is deterministic/template-based; optional OpenAI mode can polish
wording while still grounding on strict retrieval evidence.
"""

import argparse
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


from openai import OpenAI


def _load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"JSON file not found: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _dedupe_keep_order(values: list[str]) -> list[str]:
    # Preserve original order while removing repeats from noisy extraction.
    seen: set[str] = set()
    out: list[str] = []
    for value in values:
        key = value.strip().lower()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(value.strip())
    return out


def _build_citations(hits: list[dict[str, Any]]) -> list[dict[str, Any]]:
    # Keep citation schema compact and stable for downstream consumers/UI.
    citations: list[dict[str, Any]] = []
    for hit in hits:
        citations.append(
            {
                "chunk_id": hit.get("chunk_id"),
                "citation_anchor": hit.get("citation_anchor"),
                "page_start": hit.get("page_start"),
                "page_end": hit.get("page_end"),
                "section": hit.get("section"),
                "score": hit.get("score"),
                "confidence": hit.get("confidence"),
                "content_preview": hit.get("content_preview"),
            }
        )
    return citations


def _clean_sentence(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_pricing_summary(phase2: dict[str, Any]) -> dict[str, Any]:
    # Pull pricing-relevant lines for deterministic narrative construction.
    entities = phase2.get("entities", {})
    pricing_records = entities.get("pricing", [])
    model_records = entities.get("engagement_models", [])

    models = _dedupe_keep_order([str(item.get("value", "")).strip() for item in model_records])

    pricing_text = [str(item.get("value", "")).strip() for item in pricing_records]
    charge_lines: list[str] = []
    for line in pricing_text:
        lower = line.lower()
        if any(k in lower for k in ("custom-quoted", "lakhs", "crores", "lump-sum", "retainer", "daily", "weekly")):
            charge_lines.append(_clean_sentence(line))
    charge_lines = _dedupe_keep_order(charge_lines)

    return {
        "engagement_models": models,
        "charge_notes": charge_lines[:6],
    }


def _extract_services_summary(phase2: dict[str, Any], hits: list[dict[str, Any]]) -> dict[str, Any]:
    entities = phase2.get("entities", {})
    service_records = entities.get("services", [])
    services = _dedupe_keep_order([str(item.get("value", "")).strip() for item in service_records])

    capability_lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("key capabilities", "what we do", "dashboard", "predict", "forecast", "alerts")):
                capability_lines.append(clean)
    capability_lines = _dedupe_keep_order(capability_lines)

    return {
        "services": services,
        "capability_notes": capability_lines[:5],
    }


def _extract_delivery_summary(hits: list[dict[str, Any]]) -> list[str]:
    lines: list[str] = []
    for hit in hits:
        preview = str(hit.get("content_preview", ""))
        for sentence in re.split(r"(?<=[.!?])\s+", preview):
            clean = _clean_sentence(sentence)
            lower = clean.lower()
            if any(k in lower for k in ("data ingestion", "pipeline", "machine learning", "security", "dashboard", "alerts")):
                lines.append(clean)
    return _dedupe_keep_order(lines)[:6]


def _compose_answer(
    query_result: dict[str, Any],
    phase2: dict[str, Any],
) -> dict[str, Any]:
    # Compose by intent so strict mode behavior stays deterministic and testable.
    hits = query_result.get("hits", [])
    entity_filter = query_result.get("entity_filter")

    if query_result.get("abstain", False):
        return {
            "status": "abstain",
            "answer": "I cannot provide a reliable answer under strict mode for this query.",
            "reason": query_result.get("abstain_reason", "No qualifying evidence."),
            "citations": [],
            "structured_answer": {},
        }

    citations = _build_citations(hits)

    if entity_filter == "pricing":
        summary = _extract_pricing_summary(phase2)
        models = summary.get("engagement_models", [])
        charge_notes = summary.get("charge_notes", [])

        lines: list[str] = []
        if models:
            lines.append("Engagement models offered: " + ", ".join(models) + ".")
        if charge_notes:
            lines.append("Charges are described as follows: " + " ".join(charge_notes[:3]))
        lines.append("Final commercials are confirmed after discovery/scoping, so exact fixed price cards are not listed in the handbook.")

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from pricing-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "engagement_models": models,
                "charge_notes": charge_notes,
            },
        }

    if entity_filter == "services":
        summary = _extract_services_summary(phase2, hits)
        services = summary.get("services", [])
        capability_notes = summary.get("capability_notes", [])

        lines = []
        if services:
            lines.append("Services provided: " + ", ".join(services) + ".")
        if capability_notes:
            lines.append("Capabilities include: " + " ".join(capability_notes[:3]))

        return {
            "status": "answered",
            "answer": " ".join(lines),
            "reason": "Answered from service-tagged chunks under strict metadata filtering.",
            "citations": citations,
            "structured_answer": {
                "services": services,
                "capability_notes": capability_notes,
            },
        }

    if entity_filter == "delivery_approach":
        approach_notes = _extract_delivery_summary(hits)
        answer = "Technical approach highlights: " + " ".join(approach_notes) if approach_notes else "No clear delivery approach lines were found."
        return {
            "status": "answered" if approach_notes else "abstain",
            "answer": answer,
            "reason": "Answered from delivery-approach-tagged chunks." if approach_notes else "Insufficient approach evidence.",
            "citations": citations if approach_notes else [],
            "structured_answer": {"approach_notes": approach_notes},
        }

    if entity_filter == "sla_terms":
        sla_lines = []
        for hit in hits:
            preview = str(hit.get("content_preview", ""))
            for sentence in re.split(r"(?<=[.!?])\s+", preview):
                clean = _clean_sentence(sentence)
                lower = clean.lower()
                if any(k in lower for k in ("sla", "support", "monitoring", "response", "alerts")):
                    sla_lines.append(clean)
        sla_lines = _dedupe_keep_order(sla_lines)

        if not sla_lines:
            return {
                "status": "abstain",
                "answer": "I cannot provide a reliable SLA answer from available strict evidence.",
                "reason": "SLA intent matched, but no sufficiently explicit SLA lines were found.",
                "citations": [],
                "structured_answer": {},
            }

        return {
            "status": "answered",
            "answer": "SLA/support related points: " + " ".join(sla_lines[:4]),
            "reason": "Answered from SLA-tagged chunks.",
            "citations": citations,
            "structured_answer": {"sla_notes": sla_lines},
        }

    return {
        "status": "abstain",
        "answer": "I cannot answer this query under strict policy.",
        "reason": "Unsupported query intent for deterministic Phase 5 templates.",
        "citations": [],
        "structured_answer": {},
    }

def _synthesize_with_openai(
    query_text: str,
    deterministic_response: dict[str, Any],
    retrieval: dict[str, Any],
    model: str
) -> dict[str, Any]:
    # OpenAI synthesis is optional and never overrides strict abstain decisions.
    if deterministic_response.get("status") == "abstain":
        return {
            "status": "skipped",
            "answer": "",
            "reason": "Skipped OpenAI generation because strict-mode response abstained.",
            "model": model,
        }


    if not api_key:
        return {
            "status": "unavailable",
            "answer": "",
            "reason": "OPENAI_API_KEY was not found in environment or .env file.",
            "model": model,
        }

    client = OpenAI(api_key=api_key)

    citations = deterministic_response.get("citations", [])
    citation_lines = []
    for item in citations:
        anchor = item.get("citation_anchor", "")
        preview = item.get("content_preview", "")
        citation_lines.append(f"- {anchor}: {preview}")

    prompt = (
        # Prompt explicitly limits generation to retrieved evidence.
        "You are generating a strict, citation-grounded answer for a consulting handbook retrieval system. "
        "Use only the provided evidence snippets and do not invent details. "
        "If evidence does not provide exact numeric prices, explicitly state that pricing is custom-quoted. "
        "Return plain text, 4-8 sentences max, and include citation anchors inline like (page:16).\n\n"
        f"User query: {query_text}\n\n"
        f"Deterministic draft answer:\n{deterministic_response.get('answer', '')}\n\n"
        "Evidence snippets:\n"
        + "\n".join(citation_lines)
    )

    try:
        completion = client.responses.create(
            model=model,
            input=prompt,
            temperature=0,
        )
        text = completion.output_text.strip()
        return {
            "status": "answered",
            "answer": text,
            "reason": "Generated by OpenAI from strict retrieval evidence.",
            "model": model,
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "status": "error",
            "answer": "",
            "reason": f"OpenAI generation failed: {type(exc).__name__}",
            "model": model,
        }


def run_phase5(
    query_text: str,
    persist_dir: Path,
    collection_name: str,
    phase2_entities_file: Path,
    top_k: int,
    min_confidence: float,
    strict_mode: bool,
    use_openai: bool,
    openai_model: str
) -> dict[str, Any]:
    # Retrieval always runs first; answer generation is downstream of that result.
    query_result = metadata_first_query(
        persist_dir=persist_dir,
        collection_name=collection_name,
        query_text=query_text,
        top_k=top_k,
        min_confidence=min_confidence,
        strict_mode=strict_mode,
    )

    phase2 = _load_json(phase2_entities_file)
    answer_payload = _compose_answer(query_result, phase2)

    openai_payload: dict[str, Any] | None = None
    if use_openai:
        openai_payload = _synthesize_with_openai(
            query_text=query_text,
            deterministic_response=answer_payload,
            retrieval=query_result,
            model=openai_model
        )

    result = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "query": query_text,
        "strict_mode": strict_mode,
        "retrieval": query_result,
        "response": answer_payload,
    }
    if openai_payload is not None:
        result["openai_response"] = openai_payload
    return result

This cell defines the path to the `structured_entities.json` file, which is an output from Phase 2 and contains the extracted entities from the PDF document. This file is a key input for Phase 5's answer generation.

In [34]:
strcutured_entitles = Path("/content/extraction_output/phase2_structured_entities.json")

This cell executes the `run_phase5` function, which is responsible for generating an answer to the `query_text` based on the retrieved chunks and structured entities. It uses the specified `persist_dir`, `collection_name`, and `structured_entities` file. It also configures whether to use OpenAI for synthesis and which model to use. Finally, it prints the generated answer.

In [35]:
result = run_phase5(
        query_text=query_text,
        persist_dir=persist_dir,
        collection_name=collection_name,
        phase2_entities_file=strcutured_entitles,
        top_k=5,
        min_confidence=0.85,
        strict_mode="true",
        use_openai="true",
        openai_model="gpt-4o-mini"
    )
print(result["response"].get("answer"))

Services provided: Warranty Analytics, Supply-Chain Risk Prediction, Dealer & Field Service Intelligence.


This cell defines the `answer_file` path, which is where the final generated answer will be saved in JSON format. It creates a `Path` object for `answer.json` within the `extraction_output` directory.

In [36]:
answer_file = Path("/content/extraction_output/answer.json")

This cell saves the `result` (the generated answer and retrieval details from `run_phase5`) to the `answer_file` in JSON format. It ensures the output directory exists before writing the file, making the answer persistent.

In [37]:
out_file = Path(answer_file)
out_file.parent.mkdir(parents=True, exist_ok=True)
with out_file.open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)


This cell installs the necessary Python libraries for market ingestion: `agno`, `duckduckgo-search`, and `ddgs`.

In [38]:
!pip install -q agno duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 63.7 MB/s eta 0:00:00


This block defines the market ingestion pipeline for Indian automotive research. It includes data structures (`MarketRecord`) and functions for collecting web records from various sources (`collect_web_records`) and market signals from Yahoo Finance (`collect_market_signals`). It also provides utility functions for data normalization, hashing, company guessing, and topic tagging. The `write_phase1_market_artifacts` function saves the collected data, and `build_market_chunks` prepares the data for chunking. The `run_market_ingestion` function orchestrates the entire market data collection and indexing process.

In [39]:
"""Market ingestion pipeline for Indian automotive research.

This module only ingests metadata/snippets from search/news sources and market
signals from yfinance. It does not scrape blocked or paywalled pages.
"""

import argparse
import hashlib
import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import yfinance as yf
from ddgs import DDGS

SOURCE_CONFIGS: list[dict[str, str]] = [
    {"name": "NSE India", "domain": "nseindia.com", "category": "financial"},
    {"name": "BSE India", "domain": "bseindia.com", "category": "financial"},
    {"name": "Moneycontrol", "domain": "moneycontrol.com", "category": "financial"},
    {"name": "Screener", "domain": "screener.in", "category": "financial"},
    {"name": "IBEF", "domain": "ibef.org", "category": "industry"},
    {"name": "SIAM", "domain": "siam.in", "category": "industry"},
    {"name": "ACMA", "domain": "acma.in", "category": "industry"},
    {"name": "Economic Times Auto", "domain": "economictimes.indiatimes.com", "category": "news"},
    {"name": "Autocar Professional", "domain": "autocarpro.in", "category": "news"},
    {"name": "Google News", "domain": "news.google.com", "category": "news"},
]

COMPANY_CATALOG: list[dict[str, str]] = [
    {"name": "Maruti Suzuki", "ticker": "MARUTI.NS", "company_type": "OEM"},
    {"name": "Tata Motors", "ticker": "TATAMOTORS.NS", "company_type": "OEM"},
    {"name": "Mahindra & Mahindra", "ticker": "M&M.NS", "company_type": "OEM"},
    {"name": "Ashok Leyland", "ticker": "ASHOKLEY.NS", "company_type": "OEM"},
    {"name": "Bajaj Auto", "ticker": "BAJAJ-AUTO.NS", "company_type": "OEM"},
    {"name": "Eicher Motors", "ticker": "EICHERMOT.NS", "company_type": "OEM"},
    {"name": "TVS Motor", "ticker": "TVSMOTOR.NS", "company_type": "OEM"},
    {"name": "Bosch", "ticker": "BOSCHLTD.NS", "company_type": "Tier-1 Supplier"},
    {"name": "Motherson", "ticker": "MOTHERSON.NS", "company_type": "Tier-1 Supplier"},
    {"name": "Bharat Forge", "ticker": "BHARATFORG.NS", "company_type": "Component Manufacturer"},
    {"name": "Sona BLW", "ticker": "SONACOMS.NS", "company_type": "Component Manufacturer"},
    {"name": "UNO Minda", "ticker": "UNOMINDA.NS", "company_type": "Component Manufacturer"},
    {"name": "Exide Industries", "ticker": "EXIDEIND.NS", "company_type": "Component Manufacturer"},
    {"name": "Amara Raja", "ticker": "ARE&M.NS", "company_type": "Component Manufacturer"},
]

TICKER_ALIASES: dict[str, list[str]] = {
    "TATAMOTORS.NS": ["TATAMTRDVR.NS", "500570.BO"],
    "M&M.NS": ["500520.BO"],
    "ARE&M.NS": ["AMARAJABAT.NS", "500008.BO"],
    "MOTHERSON.NS": ["MOTHERSUMI.NS", "517334.BO"],
}


@dataclass
class MarketRecord:
    record_id: str
    source_name: str
    source_domain: str
    source_category: str
    source_tier: str
    title: str
    url: str
    snippet: str
    published: str | None
    company_name: str | None
    company_type: str | None
    topic_tags: list[str]
    access_level: str
    compliance_status: str
    fetched_at: str


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _norm(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "")
    return text.strip()


def _hash(*parts: str) -> str:
    material = "||".join(parts)
    return hashlib.sha256(material.encode("utf-8")).hexdigest()[:16]


def _guess_company(snippet: str, title: str) -> tuple[str | None, str | None]:
    combined = f"{title} {snippet}".lower()
    for company in COMPANY_CATALOG:
        if company["name"].lower() in combined:
            return company["name"], company["company_type"]
    return None, None


def _topic_tags(text: str) -> list[str]:
    lowered = text.lower()
    tags: list[str] = []
    keys = {
        "ev": ("ev", "electric", "battery"),
        "policy": ("policy", "regulation", "fame", "pli"),
        "exports": ("export", "overseas"),
        "demand": ("sales", "demand", "volume"),
        "margin": ("margin", "ebitda", "profit"),
        "capacity": ("capacity", "plant", "manufacturing"),
    }
    for tag, hints in keys.items():
        if any(hint in lowered for hint in hints):
            tags.append(tag)
    if not tags:
        tags.append("market_analysis")
    return tags


def _extract_domain(url: str) -> str:
    parsed = urlparse(url)
    return parsed.netloc.lower().replace("www.", "")


def _source_tier(category: str) -> str:
    if category in {"financial", "industry"}:
        return "Tier1"
    return "Tier2"


def collect_web_records(topic: str, per_source_limit: int = 8) -> tuple[list[MarketRecord], list[dict[str, str]]]:
    records: list[MarketRecord] = []
    skipped: list[dict[str, str]] = []

    with DDGS() as ddgs:
        for source in SOURCE_CONFIGS:
            query = f"{topic} site:{source['domain']}"
            try:
                results = list(ddgs.text(query, max_results=per_source_limit))
            except Exception as exc:  # noqa: BLE001
                skipped.append(
                    {
                        "source": source["name"],
                        "reason": f"search_failed:{type(exc).__name__}",
                        "query": query,
                    }
                )
                continue

            if not results:
                skipped.append(
                    {
                        "source": source["name"],
                        "reason": "no_results",
                        "query": query,
                    }
                )
                continue

            for item in results:
                url = _norm(str(item.get("href", "")))
                title = _norm(str(item.get("title", "")))
                body = _norm(str(item.get("body", "")))
                if not url or not title:
                    continue

                actual_domain = _extract_domain(url)
                if source["domain"] not in actual_domain:
                    # Strict source policy: keep only exact source-family matches.
                    skipped.append(
                        {
                            "source": source["name"],
                            "reason": "domain_mismatch",
                            "url": url,
                        }
                    )
                    continue

                company_name, company_type = _guess_company(body, title)
                tags = _topic_tags(f"{title} {body}")

                rec = MarketRecord(
                    record_id=_hash(source["name"], url),
                    source_name=source["name"],
                    source_domain=actual_domain,
                    source_category=source["category"],
                    source_tier=_source_tier(source["category"]),
                    title=title,
                    url=url,
                    snippet=body,
                    published=None,
                    company_name=company_name,
                    company_type=company_type,
                    topic_tags=tags,
                    access_level="Public metadata/search snippet",
                    compliance_status="compliant",
                    fetched_at=_now_iso(),
                )
                records.append(rec)

    deduped: dict[str, MarketRecord] = {}
    for rec in records:
        deduped[rec.record_id] = rec
    return list(deduped.values()), skipped


def _history_for_ticker_with_fallback(primary_ticker: str, period: str) -> tuple[Any, str | None]:
    candidates = [primary_ticker] + TICKER_ALIASES.get(primary_ticker, [])
    for ticker in candidates:
        try:
            history = yf.Ticker(ticker).history(period=period)
            if not history.empty:
                return history, ticker
        except Exception:  # noqa: BLE001
            continue
    return None, None


def collect_market_signals(period: str = "1mo") -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    signals: list[dict[str, Any]] = []
    skipped: list[dict[str, str]] = []
    for company in COMPANY_CATALOG:
        primary_ticker = company["ticker"]
        try:
            history, resolved_ticker = _history_for_ticker_with_fallback(primary_ticker=primary_ticker, period=period)
            if history is None or resolved_ticker is None:
                skipped.append(
                    {
                        "company_name": company["name"],
                        "ticker": primary_ticker,
                        "reason": "no_market_data",
                    }
                )
                continue

            close_first = float(history["Close"].iloc[0])
            close_last = float(history["Close"].iloc[-1])
            pct_change = round(((close_last - close_first) / max(close_first, 0.0001)) * 100, 2)

            signals.append(
                {
                    "company_name": company["name"],
                    "company_type": company["company_type"],
                    "ticker": resolved_ticker,
                    "ticker_primary": primary_ticker,
                    "period": period,
                    "close_start": round(close_first, 2),
                    "close_end": round(close_last, 2),
                    "percent_change": pct_change,
                    "signal": "positive_momentum" if pct_change > 0 else "negative_momentum",
                    "source": "yfinance",
                    "used_fallback_ticker": resolved_ticker != primary_ticker,
                    "fetched_at": _now_iso(),
                }
            )
        except Exception:  # noqa: BLE001
            skipped.append(
                {
                    "company_name": company["name"],
                    "ticker": primary_ticker,
                    "reason": "signal_exception",
                }
            )
            continue
    return signals, skipped


def write_phase1_market_artifacts(
    records: list[MarketRecord],
    signals: list[dict[str, Any]],
    skipped: list[dict[str, str]],
    signal_skipped: list[dict[str, str]],
    out_dir: Path,
) -> tuple[Path, Path, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    records_path = out_dir / "market_records.jsonl"
    signals_path = out_dir / "market_signals.json"
    report_path = out_dir / "fetch_report.json"

    with records_path.open("w", encoding="utf-8") as handle:
        for rec in records:
            handle.write(json.dumps(asdict(rec), ensure_ascii=False) + "\n")

    with signals_path.open("w", encoding="utf-8") as handle:
        json.dump(signals, handle, indent=2, ensure_ascii=False)

    report = {
        "generated_at": _now_iso(),
        "records": len(records),
        "signals": len(signals),
        "skipped": len(skipped),
        "skipped_details": skipped,
        "signals_skipped": len(signal_skipped),
        "signals_skipped_details": signal_skipped,
        "sources": sorted({rec.source_name for rec in records}),
    }
    with report_path.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)

    return records_path, signals_path, report_path


def _market_section(rec: MarketRecord) -> str:
    if rec.source_category == "financial":
        return "company_performance"
    if rec.source_category == "industry":
        return "industry_trends"
    return "market_news"


def build_market_chunks(records: list[MarketRecord], doc_id: str = "india_auto_market_v1") -> list[dict[str, Any]]:
    chunks: list[dict[str, Any]] = []
    for idx, rec in enumerate(records, start=1):
        content = _norm(f"{rec.title}. {rec.snippet}")
        if not content:
            continue

        entity_tags = ["market_analysis"]
        entity_tags.extend(rec.topic_tags)
        if rec.company_type:
            entity_tags.append(rec.company_type.lower().replace(" ", "_"))

        chunk = {
            "chunk_id": f"{doc_id}_c{idx:05d}",
            "doc_id": doc_id,
            "content": content,
            "content_type": "text",
            "section": _market_section(rec),
            "subsection": rec.source_name.lower().replace(" ", "_"),
            "page_start": 0,
            "page_end": 0,
            "citation_anchor": rec.url,
            "has_tables": False,
            "table_count": 0,
            "chunk_length_chars": len(content),
            "source_hash": _hash(rec.url, rec.title),
            "confidence": 0.78 if rec.source_tier == "Tier2" else 0.88,
            "entity_tags": sorted(set(entity_tags)),
            "source_name": rec.source_name,
            "source_category": rec.source_category,
            "source_tier": rec.source_tier,
            "company_name": rec.company_name or "",
            "company_type": rec.company_type or "",
            "published": rec.published or "",
            "fetched_at": rec.fetched_at,
        }
        chunks.append(chunk)
    return chunks


def write_market_chunk_artifacts(chunks: list[dict[str, Any]], out_dir: Path) -> tuple[Path, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    chunks_path = out_dir / "chunks.jsonl"
    report_path = out_dir / "chunk_report.json"

    with chunks_path.open("w", encoding="utf-8") as handle:
        for chunk in chunks:
            handle.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    section_counts: dict[str, int] = {}
    for chunk in chunks:
        section = str(chunk.get("section", "general"))
        section_counts[section] = section_counts.get(section, 0) + 1

    report = {
        "generated_at": _now_iso(),
        "total_chunks": len(chunks),
        "sections": section_counts,
        "avg_chunk_chars": round(sum(int(c.get("chunk_length_chars", 0)) for c in chunks) / max(len(chunks), 1), 2),
    }

    with report_path.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)

    return chunks_path, report_path


def run_market_ingestion(
    topic: str,
    phase1_market_dir: Path,
    phase3_market_dir: Path,
    phase4_persist_dir: Path,
    collection_name: str,
    per_source_limit: int,
) -> dict[str, Any]:
    records, skipped = collect_web_records(topic=topic, per_source_limit=per_source_limit)
    signals, signal_skipped = collect_market_signals(period="1mo")
    records_path, signals_path, fetch_report_path = write_phase1_market_artifacts(records, signals, skipped, signal_skipped, phase1_market_dir)

    chunks = build_market_chunks(records=records)
    chunks_path, chunk_report_path = write_market_chunk_artifacts(chunks, phase3_market_dir)

    index_report = index_chunks(
        chunks_file=chunks_path,
        persist_dir=phase4_persist_dir,
        collection_name=collection_name,
    )

    return {
        "topic": topic,
        "phase1_records_file": str(records_path),
        "phase1_signals_file": str(signals_path),
        "phase1_report_file": str(fetch_report_path),
        "phase3_chunks_file": str(chunks_path),
        "phase3_report_file": str(chunk_report_path),
        "phase4_index_report": index_report,
    }


This cell defines the output `path` for market ingestion and the `topic` for market research, setting up the context for the subsequent market data collection.

In [40]:
path = Path("/content/extraction_output")
topic = "Indian automotive industry market trends"

This cell executes the `run_market_ingestion` function, which fetches market data and signals for the specified topic, then indexes this information into a ChromaDB collection. It also saves a detailed report of the ingestion process.

In [41]:
result = run_market_ingestion(
        topic=topic,
        phase1_market_dir=path,
        phase3_market_dir=path,
        phase4_persist_dir=Path("/content/extraction_output/chroma_db"),
        collection_name="india_auto_market_chunks",
        per_source_limit=8,
    )
out_file = Path("/content/extraction_output/market_index_report.json")
with out_file.open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2, ensure_ascii=False)

print("Market ingestion complete")
print(f"- Topic: {result['topic']}")
print(f"- Records: {result['phase1_records_file']}")
print(f"- Chunks: {result['phase3_chunks_file']}")
print(f"- Collection: {result['phase4_index_report']['collection_name']}")
print(f"- Output: {out_file}")

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$500570.BO: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


Market ingestion complete
- Topic: Indian automotive industry market trends
- Records: /content/extraction_output/market_records.jsonl
- Chunks: /content/extraction_output/chunks.jsonl
- Collection: india_auto_market_chunks
- Output: /content/extraction_output/market_index_report.json


This block defines the `MarketResearchOrchestrator` class, which orchestrates a multi-agent market research process using the Agno framework and OpenAI models. It sets up agents for research, validation, and reporting, and includes functions for retrieving consultancy services, market evidence, running research and validation steps, and generating a final markdown report.

In [42]:
"""Agno multi-agent market research orchestration for Indian automotive industry."""

import json
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from agno.agent import Agent
from agno.models.openai import OpenAIChat

@dataclass
class AgentRunConfig:
    persist_dir: Path = Path("/content/extraction_output/chroma_db")
    handbook_collection: str = "xyz_handbook_chunks"
    market_collection: str = "india_auto_market_chunks"
    phase2_entities_file: Path = Path("/content/extraction_output/phase2_structured_entities.json")
    top_k: int = 8
    min_confidence: float = 0.74
    openai_model: str = "gpt-4o-mini"


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _extract_json_block(text: str) -> dict[str, Any] | None:
    cleaned = text.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def _to_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _sanitize_research_payload(payload: dict[str, Any]) -> dict[str, Any]:
    organizations_in = payload.get("organizations", [])
    opportunities_in = payload.get("opportunities", [])

    organizations: list[dict[str, Any]] = []
    for item in organizations_in if isinstance(organizations_in, list) else []:
        if not isinstance(item, dict):
            continue
        name = _safe_text(item.get("name")) or "Unknown Organization"
        category = _safe_text(item.get("category")) or "Unknown"
        priority_score = int(max(0, min(100, round(_to_float(item.get("priority_score_0_to_100"), 0.0)))))
        fit_explanation = _safe_text(item.get("fit_explanation"))

        citations_in = item.get("supporting_citations", [])
        citations: list[dict[str, str]] = []
        if isinstance(citations_in, list):
            for citation in citations_in:
                if isinstance(citation, dict):
                    citations.append(
                        {
                            "chunk_id": _safe_text(citation.get("chunk_id")),
                            "citation_anchor": _safe_text(citation.get("citation_anchor")),
                        }
                    )

        organizations.append(
            {
                "name": name,
                "category": category,
                "priority_score_0_to_100": priority_score,
                "fit_explanation": fit_explanation,
                "supporting_citations": citations,
            }
        )

    opportunities: list[dict[str, Any]] = []
    for item in opportunities_in if isinstance(opportunities_in, list) else []:
        if not isinstance(item, dict):
            continue
        citations_in = item.get("supporting_citations", [])
        citations: list[dict[str, str]] = []
        if isinstance(citations_in, list):
            for citation in citations_in:
                if isinstance(citation, dict):
                    citations.append(
                        {
                            "chunk_id": _safe_text(citation.get("chunk_id")),
                            "citation_anchor": _safe_text(citation.get("citation_anchor")),
                        }
                    )
        opportunities.append(
            {
                "description": _safe_text(item.get("description")),
                "category": _safe_text(item.get("category")) or "General",
                "supporting_citations": citations,
            }
        )

    trend_summary_raw = payload.get("trend_summary", [])
    if isinstance(trend_summary_raw, list):
        trend_summary = [_safe_text(item) for item in trend_summary_raw if _safe_text(item)]
    else:
        one_line = _safe_text(trend_summary_raw)
        trend_summary = [one_line] if one_line else []

    assumptions_raw = payload.get("assumptions", [])
    if isinstance(assumptions_raw, list):
        assumptions = [_safe_text(item) for item in assumptions_raw if _safe_text(item)]
    elif isinstance(assumptions_raw, dict):
        assumptions = [f"{_safe_text(k)}: {_safe_text(v)}" for k, v in assumptions_raw.items() if _safe_text(k) or _safe_text(v)]
    else:
        one_line = _safe_text(assumptions_raw)
        assumptions = [one_line] if one_line else []

    return {
        "organizations": organizations,
        "opportunities": opportunities,
        "trend_summary": trend_summary,
        "assumptions": assumptions,
    }


def _sanitize_validation_payload(payload: dict[str, Any], fallback_organizations: list[dict[str, Any]]) -> dict[str, Any]:
    warnings_raw = payload.get("warnings", [])
    warnings = [_safe_text(item) for item in warnings_raw] if isinstance(warnings_raw, list) else [_safe_text(warnings_raw)]

    review_raw = payload.get("must_review_items", [])
    must_review_items: list[dict[str, str]] = []
    if isinstance(review_raw, list):
        for item in review_raw:
            if isinstance(item, dict):
                review_item = _safe_text(item.get("item"))
                review_reason = _safe_text(item.get("reason"))
                if not review_item and not review_reason:
                    continue
                must_review_items.append(
                    {
                        "item": review_item,
                        "reason": review_reason,
                    }
                )

    organizations_raw = payload.get("validated_organizations", fallback_organizations)
    validated_organizations: list[dict[str, Any]] = []
    if isinstance(organizations_raw, list):
        for item in organizations_raw:
            if isinstance(item, dict):
                validated_organizations.append(
                    {
                        "name": _safe_text(item.get("name")) or "Unknown Organization",
                        "fit_explanation": _safe_text(item.get("fit_explanation")),
                        "supporting_citations": item.get("supporting_citations", []),
                    }
                )

    overall_confidence = _to_float(payload.get("overall_confidence"), 0.5)
    overall_confidence = max(0.0, min(1.0, overall_confidence))

    return {
        "overall_confidence": round(overall_confidence, 2),
        "warnings": [w for w in warnings if w],
        "validated_organizations": validated_organizations,
        "must_review_items": must_review_items,
    }


def _apply_confidence_gates(
    validation_payload: dict[str, Any],
    research_payload: dict[str, Any],
    retrieval: dict[str, Any],
    min_citations: int = 2,
    min_confidence: float = 0.78,
) -> dict[str, Any]:
    confidence_by_chunk: dict[str, float] = {}
    for hit in retrieval.get("hits", []):
        chunk_id = _safe_text(hit.get("chunk_id"))
        if not chunk_id:
            continue
        confidence_by_chunk[chunk_id] = _to_float(hit.get("confidence"), 0.0)

    gate_results: list[dict[str, Any]] = []
    gating_failures = 0

    organizations = research_payload.get("organizations", [])
    for org in organizations if isinstance(organizations, list) else []:
        if not isinstance(org, dict):
            continue
        org_name = _safe_text(org.get("name")) or "Unknown Organization"
        citations = org.get("supporting_citations", [])
        citation_items = citations if isinstance(citations, list) else []

        high_conf_count = 0
        for citation in citation_items:
            if not isinstance(citation, dict):
                continue
            chunk_id = _safe_text(citation.get("chunk_id"))
            if confidence_by_chunk.get(chunk_id, 0.0) >= min_confidence:
                high_conf_count += 1

        passes = len(citation_items) >= min_citations and high_conf_count >= min_citations
        if not passes:
            gating_failures += 1

        gate_results.append(
            {
                "organization": org_name,
                "citation_count": len(citation_items),
                "high_confidence_citation_count": high_conf_count,
                "min_citations_required": min_citations,
                "min_confidence_required": min_confidence,
                "passes_gate": passes,
            }
        )

    warnings = list(validation_payload.get("warnings", []))
    if gating_failures > 0:
        warnings.append(
            f"{gating_failures} organization(s) failed strict citation-confidence gates."
        )

    validation_payload["warnings"] = warnings
    validation_payload["gates"] = {
        "min_citations_per_org": min_citations,
        "min_confidence_per_citation": min_confidence,
        "failed_organizations": gating_failures,
        "results": gate_results,
    }
    return validation_payload


def _group_market_hits(hits: list[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:
    grouped: dict[str, list[dict[str, Any]]] = {
        "OEM": [],
        "Tier-1 Supplier": [],
        "Component Manufacturer": [],
        "Unknown": [],
    }
    for hit in hits:
        text = f"{hit.get('content_preview', '')} {hit.get('citation_anchor', '')}".lower()
        category = "Unknown"
        if "tier-1" in text or "tier 1" in text:
            category = "Tier-1 Supplier"
        elif "component" in text or "auto parts" in text:
            category = "Component Manufacturer"
        elif any(name in text for name in ("motors", "suzuki", "mahindra", "leyland", "bajaj", "tvs")):
            category = "OEM"
        grouped[category].append(hit)
    return grouped


def _compute_simple_signals(hits: list[dict[str, Any]]) -> list[str]:
    combined = " ".join(str(hit.get("content_preview", "")) for hit in hits).lower()
    trends: list[str] = []
    if any(term in combined for term in ("ev", "electric", "battery")):
        trends.append("EV adoption and electrification continues to shape strategy.")
    if any(term in combined for term in ("export", "overseas")):
        trends.append("Export momentum is a recurring growth lever.")
    if any(term in combined for term in ("policy", "fame", "pli", "incentive")):
        trends.append("Policy and incentive shifts are material for planning and investments.")
    if any(term in combined for term in ("margin", "cost", "commodity")):
        trends.append("Margin resilience and cost optimization remain board-level priorities.")
    if not trends:
        trends.append("Mixed market outlook with company-specific execution differences.")
    return trends


def _fit_rubric() -> dict[str, str]:
    return {
        "service_alignment": "Signals where analytics/forecasting, demand planning, quality analytics, or connected operations are relevant.",
        "urgency": "Signals of immediate pressure: margin squeeze, demand volatility, recalls, compliance pressure.",
        "execution_readiness": "Signals of digital maturity and adoption readiness.",
        "commercial_potential": "Potential consulting scope breadth and depth.",
    }


class MarketResearchOrchestrator:
    def __init__(self, config: AgentRunConfig) -> None:
        self.config = config

        model = OpenAIChat(id=config.openai_model, api_key=api_key)

        self.research_agent = Agent(
            name="Indian Auto Research Agent",
            model=model,
            instructions=[
                "You are a strict market research analyst focused on Indian automotive industry.",
                "Use only provided tool outputs and citations.",
                "Return concise JSON-compatible answers when asked.",
            ],
        )
        self.validation_agent = Agent(
            name="Evidence Validation Agent",
            model=model,
            instructions=[
                "Validate evidence sufficiency, freshness, and confidence.",
                "Do not invent data. Mark low-confidence recommendations explicitly.",
            ],
        )
        self.report_agent = Agent(
            name="Market Report Agent",
            model=model,
            instructions=[
                "Produce a concise markdown report from structured validated input.",
                "Use section headers and include cited source links grouped by findings.",
            ],
        )


    def get_consultancy_services(self) -> dict[str, Any]:
        service_query = run_phase5(
            query_text="What services and capabilities are offered?",
            persist_dir=self.config.persist_dir,
            collection_name=self.config.handbook_collection,
            phase2_entities_file=self.config.phase2_entities_file,
            top_k=5,
            min_confidence=0.85,
            strict_mode=True,
            use_openai=False,
            openai_model=self.config.openai_model
        )

        pricing_query = run_phase5(
            query_text="What pricing and engagement models are available?",
            persist_dir=self.config.persist_dir,
            collection_name=self.config.handbook_collection,
            phase2_entities_file=self.config.phase2_entities_file,
            top_k=5,
            min_confidence=0.85,
            strict_mode=True,
            use_openai=False,
            openai_model=self.config.openai_model
        )

        return {
            "services_response": service_query.get("response", {}),
            "pricing_response": pricing_query.get("response", {}),
        }

    def retrieve_market_evidence(self, topic: str) -> dict[str, Any]:
        retrieval = metadata_first_query(
            persist_dir=self.config.persist_dir,
            collection_name=self.config.market_collection,
            query_text=topic,
            top_k=self.config.top_k,
            min_confidence=self.config.min_confidence,
            strict_mode=False,
        )
        return retrieval

    def _run_research_step(
        self,
        topic: str,
        consultancy_context: dict[str, Any],
        market_retrieval: dict[str, Any],
    ) -> dict[str, Any]:
        grouped = _group_market_hits(market_retrieval.get("hits", []))
        trends = _compute_simple_signals(market_retrieval.get("hits", []))

        input_payload = {
            "topic": topic,
            "consultancy_context": consultancy_context,
            "grouped_market_hits": grouped,
            "computed_trends": trends,
            "rubric": _fit_rubric(),
        }

        prompt = (
            "Create candidate organizations and opportunities from the supplied payload. "
            "Return JSON with keys: organizations, opportunities, trend_summary, assumptions. "
            "Each organization item must include: name, category, priority_score_0_to_100, "
            "fit_explanation, supporting_citations."
            f"\n\nPayload:\n{json.dumps(input_payload, ensure_ascii=False)}"
        )
        result = self.research_agent.run(prompt)
        text = _safe_text(getattr(result, "content", ""))

        parsed = _extract_json_block(text)
        if parsed is not None:
            return _sanitize_research_payload(parsed)

        # Fallback keeps pipeline robust when model does not emit valid JSON.
        return {
            "organizations": [],
            "opportunities": [],
            "trend_summary": trends,
            "assumptions": ["Model output was not valid JSON; fallback payload emitted."],
            "raw_model_output": text,
        }

    def _run_validation_step(self, research_payload: dict[str, Any], retrieval: dict[str, Any]) -> dict[str, Any]:
        prompt = (
            "Validate this research payload for evidence sufficiency and confidence. "
            "Return JSON with: overall_confidence, warnings, validated_organizations, must_review_items."
            f"\n\nResearchPayload:\n{json.dumps(research_payload, ensure_ascii=False)}"
            f"\n\nRetrievalMeta:\n{json.dumps(retrieval, ensure_ascii=False)}"
        )
        result = self.validation_agent.run(prompt)
        text = _safe_text(getattr(result, "content", ""))
        parsed = _extract_json_block(text)
        fallback_payload = {
            "overall_confidence": 0.5,
            "warnings": ["Validation model output could not be parsed as JSON."],
            "validated_organizations": research_payload.get("organizations", []),
            "must_review_items": [],
            "raw_model_output": text,
        }
        if parsed is not None:
            sanitized = _sanitize_validation_payload(parsed, research_payload.get("organizations", []))
        else:
            sanitized = _sanitize_validation_payload(fallback_payload, research_payload.get("organizations", []))

        gated = _apply_confidence_gates(
            validation_payload=sanitized,
            research_payload=research_payload,
            retrieval=retrieval,
            min_citations=2,
            min_confidence=self.config.min_confidence,
        )
        if parsed is None:
            gated["raw_model_output"] = text
        return gated

    def _run_report_step(self, final_json: dict[str, Any]) -> str:
        prompt = (
            "Write a compact market research report in markdown for business stakeholders. "
            "Sections required: Executive Summary, Industry Trends, Target Organizations, "
            "Consultancy Fit Rationale, Source Notes."
            f"\n\nInput JSON:\n{json.dumps(final_json, ensure_ascii=False)}"
        )
        result = self.report_agent.run(prompt)
        return _safe_text(getattr(result, "content", ""))

    def run(self, topic: str) -> tuple[dict[str, Any], str]:
        consultancy_context = self.get_consultancy_services()
        market_retrieval = self.retrieve_market_evidence(topic)

        research_json = self._run_research_step(topic, consultancy_context, market_retrieval)
        validation_json = self._run_validation_step(research_json, market_retrieval)

        final_json = {
            "generated_at": _now_iso(),
            "topic": topic,
            "consultancy_context": consultancy_context,
            "market_retrieval": {
                "query": market_retrieval.get("query"),
                "hit_count": len(market_retrieval.get("hits", [])),
                "hits": market_retrieval.get("hits", []),
            },
            "research": research_json,
            "validation": validation_json,
            "scoring_rubric": _fit_rubric(),
        }
        markdown_report = self._run_report_step(final_json)
        return final_json, markdown_report


This block defines the company-level ingestion and indexing pipeline. It includes functions for collecting company-specific financial signals from Yahoo Finance (`collect_company_signals`) and news snippets from DuckDuckGo Search (`collect_company_news_snippets`). The `build_company_chunks` function processes this data into chunks, and `run_company_ingestion` orchestrates the entire process, including indexing the chunks into ChromaDB.

In [43]:
"""Company-level ingestion and indexing for stage-2 research."""

import argparse
import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import yfinance as yf
from ddgs import DDGS


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _norm(value: Any) -> str:
    if value is None:
        return ""
    return " ".join(str(value).strip().split())


def _hash(*parts: str) -> str:
    material = "||".join(parts)
    return hashlib.sha256(material.encode("utf-8")).hexdigest()[:16]


def _safe_float(value: Any) -> float | None:
    try:
        out = float(value)
        if math.isnan(out) or math.isinf(out):
            return None
        return out
    except (TypeError, ValueError):
        return None


def _history_with_fallback(primary_ticker: str, period: str) -> tuple[Any, str | None]:
    tickers = [primary_ticker] + TICKER_ALIASES.get(primary_ticker, [])
    for ticker in tickers:
        try:
            history = yf.Ticker(ticker).history(period=period)
            if history is not None and not history.empty:
                return history, ticker
        except Exception:  # noqa: BLE001
            continue
    return None, None


def collect_company_signals(period: str = "6mo") -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    signals: list[dict[str, Any]] = []
    skipped: list[dict[str, str]] = []

    for company in COMPANY_CATALOG:
        primary_ticker = company["ticker"]
        history, resolved_ticker = _history_with_fallback(primary_ticker=primary_ticker, period=period)
        if history is None or resolved_ticker is None:
            skipped.append(
                {
                    "company_name": company["name"],
                    "ticker": primary_ticker,
                    "reason": "no_market_data",
                }
            )
            continue

        close_start = _safe_float(history["Close"].iloc[0])
        close_end = _safe_float(history["Close"].iloc[-1])
        volume_avg = _safe_float(history["Volume"].mean())

        percent_change = None
        if close_start is not None and close_end is not None and close_start != 0:
            percent_change = round(((close_end - close_start) / close_start) * 100, 2)

        if percent_change is None:
            signal = "unknown"
        elif percent_change >= 2:
            signal = "positive_momentum"
        elif percent_change <= -2:
            signal = "negative_momentum"
        else:
            signal = "neutral_momentum"

        signals.append(
            {
                "company_name": company["name"],
                "company_type": company["company_type"],
                "ticker": resolved_ticker,
                "ticker_primary": primary_ticker,
                "period": period,
                "close_start": close_start,
                "close_end": close_end,
                "percent_change": percent_change,
                "avg_volume": round(volume_avg, 2) if volume_avg is not None else None,
                "signal": signal,
                "source": "yfinance",
                "used_fallback_ticker": resolved_ticker != primary_ticker,
                "fetched_at": _now_iso(),
            }
        )

    return signals, skipped


def collect_company_news_snippets(companies: list[dict[str, Any]], per_company_limit: int = 4) -> list[dict[str, Any]]:
    snippets: list[dict[str, Any]] = []
    with DDGS() as ddgs:
        for company in companies:
            company_name = str(company.get("name", "")).strip()
            if not company_name:
                continue

            query = f"{company_name} India automotive recent news"
            try:
                results = list(ddgs.text(query, max_results=per_company_limit))
            except Exception:  # noqa: BLE001
                continue

            for item in results:
                title = _norm(item.get("title", ""))
                url = _norm(item.get("href", ""))
                body = _norm(item.get("body", ""))
                if not title or not url:
                    continue
                snippets.append(
                    {
                        "company_name": company_name,
                        "title": title,
                        "url": url,
                        "snippet": body,
                        "fetched_at": _now_iso(),
                    }
                )

    return snippets


def build_company_chunks(signals: list[dict[str, Any]], news_snippets: list[dict[str, Any]], doc_id: str = "company_profiles_v1") -> list[dict[str, Any]]:
    news_by_company: dict[str, list[dict[str, Any]]] = {}
    for row in news_snippets:
        company = str(row.get("company_name", "")).strip()
        news_by_company.setdefault(company, []).append(row)

    chunks: list[dict[str, Any]] = []
    idx = 1
    for signal in signals:
        company_name = str(signal.get("company_name", "")).strip()
        company_type = str(signal.get("company_type", "Unknown")).strip()
        pct = signal.get("percent_change")
        momentum = str(signal.get("signal", "unknown"))
        avg_volume = signal.get("avg_volume")

        base_text = (
            f"Company: {company_name}. Type: {company_type}. "
            f"Financial momentum: {momentum}. Percent change over {signal.get('period')}: {pct}. "
            f"Average trading volume: {avg_volume}."
        )

        news_lines = []
        for item in news_by_company.get(company_name, [])[:4]:
            news_lines.append(f"{item.get('title')} ({item.get('url')})")

        if news_lines:
            base_text += " Recent news: " + " | ".join(news_lines)

        content = _norm(base_text)
        if not content:
            continue

        chunk = {
            "chunk_id": f"{doc_id}_c{idx:05d}",
            "doc_id": doc_id,
            "content": content,
            "content_type": "text",
            "section": "company_profile",
            "subsection": company_type.lower().replace(" ", "_"),
            "page_start": 0,
            "page_end": 0,
            "citation_anchor": f"company:{company_name}",
            "has_tables": False,
            "table_count": 0,
            "chunk_length_chars": len(content),
            "source_hash": _hash(company_name, str(signal.get("ticker", ""))),
            "confidence": 0.84,
            "entity_tags": sorted(
                set(
                    [
                        "company_profile",
                        company_type.lower().replace(" ", "_"),
                        company_name.lower().replace(" ", "_"),
                        str(signal.get("signal", "unknown")),
                    ]
                )
            ),
            "company_name": company_name,
            "company_type": company_type,
            "ticker": signal.get("ticker", ""),
            "percent_change": signal.get("percent_change"),
            "avg_volume": signal.get("avg_volume"),
            "fetched_at": signal.get("fetched_at"),
        }
        chunks.append(chunk)
        idx += 1

    return chunks


def run_company_ingestion(
    phase1_company_dir: Path,
    phase3_company_dir: Path,
    phase4_persist_dir: Path,
    collection_name: str,
    per_company_news_limit: int,
) -> dict[str, Any]:
    phase1_company_dir.mkdir(parents=True, exist_ok=True)
    phase3_company_dir.mkdir(parents=True, exist_ok=True)

    signals, signal_skipped = collect_company_signals(period="6mo")
    news_snippets = collect_company_news_snippets(COMPANY_CATALOG, per_company_limit=per_company_news_limit)
    chunks = build_company_chunks(signals=signals, news_snippets=news_snippets)

    signals_path = phase1_company_dir / "company_signals.json"
    news_path = phase1_company_dir / "company_news.json"
    report_path = phase1_company_dir / "company_ingestion_report.json"

    with signals_path.open("w", encoding="utf-8") as handle:
        json.dump(signals, handle, indent=2, ensure_ascii=False, allow_nan=False)

    with news_path.open("w", encoding="utf-8") as handle:
        json.dump(news_snippets, handle, indent=2, ensure_ascii=False)

    chunks_path = phase3_company_dir / "company_chunks.jsonl"
    with chunks_path.open("w", encoding="utf-8") as handle:
        for chunk in chunks:
            handle.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    index_report = index_chunks(
        chunks_file=chunks_path,
        persist_dir=phase4_persist_dir,
        collection_name=collection_name,
    )

    report = {
        "generated_at": _now_iso(),
        "signals": len(signals),
        "signals_skipped": len(signal_skipped),
        "signals_skipped_details": signal_skipped,
        "news_snippets": len(news_snippets),
        "chunks": len(chunks),
        "collection": collection_name,
        "index_report": index_report,
    }

    with report_path.open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)

    return {
        "signals_file": str(signals_path),
        "news_file": str(news_path),
        "chunks_file": str(chunks_path),
        "report_file": str(report_path),
        "index_report": index_report,
    }



This cell executes the `run_company_ingestion` function, which collects and indexes company-specific financial signals and news snippets into a ChromaDB collection. It saves a report summarizing the ingestion process.

In [44]:
    result = run_company_ingestion(
        phase1_company_dir=Path("/content/extraction_output/"),
        phase3_company_dir=Path("/content/extraction_output/"),
        phase4_persist_dir=Path("/content/extraction_output/chroma_db"),
        collection_name="company_profiles_chunks",
        per_company_news_limit=4,
    )

    out_file = Path("/content/extraction_output/company_index_report.json")
    with out_file.open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2, ensure_ascii=False)

    print("Company ingestion complete")
    print(f"- Signals: {result['signals_file']}")
    print(f"- Chunks: {result['chunks_file']}")
    print(f"- Collection: {result['index_report']['collection_name']}")
    print(f"- Output: {out_file}")


ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$500570.BO: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


Company ingestion complete
- Signals: /content/extraction_output/company_signals.json
- Chunks: /content/extraction_output/company_chunks.jsonl
- Collection: company_profiles_chunks
- Output: /content/extraction_output/company_index_report.json


This block defines the `CompanyResearchOrchestrator` class, which uses Agno agents and OpenAI models to perform detailed company profiling. It includes functions for retrieving market and company-specific evidence from ChromaDB and orchestrates the process of building comprehensive company profiles based on this evidence.

In [45]:
"""Agno Company Research agent for per-company profiling."""

import json
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from agno.agent import Agent
from agno.models.openai import OpenAIChat

def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _extract_json_block(text: str) -> dict[str, Any] | None:
    cleaned = text.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def _to_float(value: Any, default: float | None = None) -> float | None:
    try:
        out = float(value)
        return out
    except (TypeError, ValueError):
        return default


def _load_json(path: Path) -> Any:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _company_catalog_map() -> dict[str, dict[str, str]]:
    mapped: dict[str, dict[str, str]] = {}
    for row in COMPANY_CATALOG:
        mapped[row["name"].lower()] = row
    return mapped


def _sanitize_profile(payload: dict[str, Any], fallback_company: dict[str, str], citations: list[dict[str, Any]]) -> dict[str, Any]:
    name = _safe_text(payload.get("company_name")) or fallback_company.get("name", "Unknown Company")
    ctype = _safe_text(payload.get("company_type")) or fallback_company.get("company_type", "Unknown")

    products_raw = payload.get("products_services", [])
    products_services = [_safe_text(x) for x in products_raw] if isinstance(products_raw, list) else []

    locations_raw = payload.get("manufacturing_locations", [])
    manufacturing_locations = [_safe_text(x) for x in locations_raw] if isinstance(locations_raw, list) else []

    challenges_raw = payload.get("business_challenges", [])
    business_challenges = [_safe_text(x) for x in challenges_raw] if isinstance(challenges_raw, list) else []

    news_raw = payload.get("recent_news", [])
    recent_news: list[dict[str, str]] = []
    if isinstance(news_raw, list):
        for item in news_raw:
            if isinstance(item, dict):
                title = _safe_text(item.get("title"))
                url = _safe_text(item.get("url"))
                if title or url:
                    recent_news.append({"title": title, "url": url})

    return {
        "company_name": name,
        "company_type": ctype,
        "company_overview": _safe_text(payload.get("company_overview")),
        "products_services": products_services,
        "manufacturing_locations": manufacturing_locations,
        "business_growth": _safe_text(payload.get("business_growth")),
        "financial_highlights": _safe_text(payload.get("financial_highlights")),
        "recent_news": recent_news,
        "business_challenges": business_challenges,
        "ticker": fallback_company.get("ticker", ""),
        "market_cap_bucket": _safe_text(payload.get("market_cap_bucket")) or "unknown",
        "revenue_trend": _safe_text(payload.get("revenue_trend")) or "unknown",
        "citations": citations,
        "profile_confidence": max(0.0, min(1.0, _to_float(payload.get("profile_confidence"), 0.72) or 0.72)),
        "generated_at": _now_iso(),
    }


@dataclass
class CompanyResearchConfig:
    persist_dir: Path = Path("outputs/phase4/chroma_db")
    market_collection: str = "india_auto_market_chunks"
    company_collection: str = "company_profiles_chunks"
    company_signals_file: Path = Path("outputs/phase1_company/company_signals.json")
    openai_model: str = "gpt-4o-mini"
    top_k_market: int = 6
    top_k_company: int = 6


class CompanyResearchOrchestrator:
    def __init__(self, config: CompanyResearchConfig) -> None:
        self.config = config

        self.catalog = _company_catalog_map()
        self.signals = _load_json(config.company_signals_file)
        self.signals_by_company = {
            str(row.get("company_name", "")).lower(): row
            for row in self.signals
            if isinstance(row, dict)
        }

        self.research_agent = Agent(
            name="Company Research Agent",
            model=OpenAIChat(id=config.openai_model, api_key=api_key),
            instructions=[
                "You are an Indian automotive company profiling analyst.",
                "Return strict JSON only using supplied evidence.",
                "Do not fabricate facts beyond evidence snippets.",
            ],
        )

    def _market_hits(self, company_name: str) -> list[dict[str, Any]]:
        result = metadata_first_query(
            persist_dir=self.config.persist_dir,
            collection_name=self.config.market_collection,
            query_text=f"{company_name} products manufacturing growth financial news challenges",
            top_k=self.config.top_k_market,
            min_confidence=0.72,
            strict_mode=False,
        )
        return result.get("hits", [])

    def _company_hits(self, company_name: str) -> list[dict[str, Any]]:
        result = metadata_first_query(
            persist_dir=self.config.persist_dir,
            collection_name=self.config.company_collection,
            query_text=f"{company_name} company profile financial signal",
            top_k=self.config.top_k_company,
            min_confidence=0.72,
            strict_mode=False,
        )
        return result.get("hits", [])

    def profile_company(self, company_name: str, company_type: str) -> dict[str, Any]:
        fallback = self.catalog.get(company_name.lower(), {"name": company_name, "company_type": company_type, "ticker": ""})
        signal = self.signals_by_company.get(company_name.lower(), {})

        market_hits = self._market_hits(company_name)
        company_hits = self._company_hits(company_name)

        citations: list[dict[str, Any]] = []
        for hit in (market_hits + company_hits):
            citations.append(
                {
                    "chunk_id": _safe_text(hit.get("chunk_id")),
                    "citation_anchor": _safe_text(hit.get("citation_anchor")),
                    "confidence": hit.get("confidence"),
                    "section": _safe_text(hit.get("section")),
                }
            )

        prompt_payload = {
            "company_name": company_name,
            "company_type": company_type,
            "financial_signal": signal,
            "market_hits": market_hits,
            "company_hits": company_hits,
            "required_fields": [
                "company_overview",
                "products_services",
                "manufacturing_locations",
                "business_growth",
                "financial_highlights",
                "recent_news",
                "business_challenges",
            ],
        }

        prompt = (
            "Create a JSON company profile from the payload. "
            "Return keys exactly: company_name, company_type, company_overview, products_services, "
            "manufacturing_locations, business_growth, financial_highlights, recent_news, business_challenges, "
            "market_cap_bucket, revenue_trend, profile_confidence."
            f"\n\nPayload:\n{json.dumps(prompt_payload, ensure_ascii=False)}"
        )

        run = self.research_agent.run(prompt)
        text = _safe_text(getattr(run, "content", ""))
        parsed = _extract_json_block(text)

        if parsed is None:
            parsed = {
                "company_name": company_name,
                "company_type": company_type,
                "company_overview": "Limited profile evidence available.",
                "products_services": [],
                "manufacturing_locations": [],
                "business_growth": "Evidence indicates evolving market participation.",
                "financial_highlights": "Refer linked citations for latest updates.",
                "recent_news": [],
                "business_challenges": ["Insufficient structured evidence in current retrieval window."],
                "profile_confidence": 0.6,
            }

        profile = _sanitize_profile(parsed, fallback, citations)
        profile["raw_model_output"] = text if parsed is None else ""
        return profile

    def run(self, candidates: list[dict[str, str]]) -> list[dict[str, Any]]:
        profiles: list[dict[str, Any]] = []
        seen: set[str] = set()

        for item in candidates:
            name = _safe_text(item.get("name"))
            ctype = _safe_text(item.get("category")) or _safe_text(item.get("company_type")) or "Unknown"
            if not name:
                continue
            key = name.lower()
            if key in seen:
                continue
            seen.add(key)
            profiles.append(self.profile_company(name, ctype))

        return profiles


This cell sets various configuration parameters for the Agno orchestration, including paths for ChromaDB persistence, collection names, `top_k` for retrieval, minimum confidence thresholds, and the OpenAI model to be used.

In [46]:
persist_dir=Path("/content/extraction_output/chroma_db")
collection_name="india_auto_market_chunks"
top_k=8
collection_name="india_auto_market_chunks"
handbook_collection="xyz_handbook_chunks"
min_confidence=0.75
market_collection="india_auto_market_chunks"
phase2_entities_file=Path("/content/extraction_output/phase2_structured_entities.json")
openai_model="gpt-4o-mini"
per_source_limit=8
topic = "Research the Indian automotive industry"

This cell orchestrates the complete market research run using the `MarketResearchOrchestrator`. It initiates market data ingestion, runs the multi-agent research process, and then saves the generated JSON report and markdown report to the specified output directory. It also creates a run manifest detailing the execution parameters and artifacts.

In [47]:
ingestion_result = run_market_ingestion(
        topic=topic,
        phase1_market_dir=path,
        phase3_market_dir=path,
        phase4_persist_dir=Path("/content/extraction_output/chroma_db"),
        collection_name="india_auto_market_chunks",
        per_source_limit=8,
    )
config = AgentRunConfig(
        persist_dir=Path("/content/extraction_output/chroma_db"),
        handbook_collection="xyz_handbook_chunks",
        market_collection="india_auto_market_chunks",
        phase2_entities_file=Path("/content/extraction_output/phase2_structured_entities.json"),
        top_k=8,
        min_confidence=0.75,
        openai_model="gpt-4o-mini",
    )

orchestrator = MarketResearchOrchestrator(config=config)
report_json, report_md = orchestrator.run(topic=topic)

out_dir = Path("/content/extraction_output/market_research")
out_dir.mkdir(parents=True, exist_ok=True)

report_json_path = Path("/content/extraction_output/market_research/report.json")
report_md_path = Path("/content/extraction_output/market_research/report.md")
manifest_path = Path("/content/extraction_output/market_research/run_manifest.json")

with report_json_path.open("w", encoding="utf-8") as handle:
   json.dump(report_json, handle, indent=2, ensure_ascii=False)

report_md_path.write_text(report_md, encoding="utf-8")

manifest = {
        "generated_at": _now_iso(),
        "topic": topic,
        "refresh_market_data": "true",
        "ingestion": ingestion_result,
        "settings": {
            "persist_dir": str(persist_dir),
            "handbook_collection": handbook_collection,
            "market_collection": market_collection,
            "phase2_entities": str(phase2_entities_file),
            "top_k": top_k,
            "min_confidence": min_confidence,
            "openai_model": openai_model,
            "per_source_limit": per_source_limit,
        },
        "artifacts": {
            "report_json": str(report_json_path),
            "report_md": str(report_md_path),
        },
    }
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print("Market research run complete")
print(f"- Topic: {topic}")
print(f"- Refresh market data: true")
print(f"- Report JSON: {report_json_path}")
print(f"- Report Markdown: {report_md_path}")
print(f"- Run manifest: {manifest_path}")

ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$500570.BO: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


Market research run complete
- Topic: Research the Indian automotive industry
- Refresh market data: true
- Report JSON: /content/extraction_output/market_research/report.json
- Report Markdown: /content/extraction_output/market_research/report.md
- Run manifest: /content/extraction_output/market_research/run_manifest.json


This block defines the `CompanyPrioritizationOrchestrator` class, which is responsible for scoring and ranking companies based on various criteria such as service alignment, urgency, readiness, and evidence depth. It uses an Agno agent and OpenAI model to generate explanations for the prioritization decisions.

In [48]:
"""Agno Prioritization agent with deterministic scoring for Top 10 selection."""

import json
import os
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from agno.agent import Agent
from agno.models.openai import OpenAIChat


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _to_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


@dataclass
class PrioritizationConfig:
    openai_model: str = "gpt-4o-mini"
    min_citations: int = 2
    min_confidence: float = 0.74


class CompanyPrioritizationOrchestrator:
    def __init__(self, config: PrioritizationConfig) -> None:
        self.config = config
        self.prioritizer_agent = Agent(
            name="Company Prioritization Agent",
            model=OpenAIChat(id=config.openai_model, api_key=api_key),
            instructions=[
                "You are a B2B consulting prioritization strategist.",
                "Use the supplied scores and evidence only.",
                "Return concise rationale text only.",
            ],
        )

    def _service_alignment_score(self, profile: dict[str, Any]) -> int:
        text = " ".join(
            [
                _safe_text(profile.get("company_overview")),
                _safe_text(profile.get("business_growth")),
                _safe_text(profile.get("financial_highlights")),
                " ".join(profile.get("business_challenges", [])) if isinstance(profile.get("business_challenges"), list) else "",
            ]
        ).lower()

        score = 8
        if any(term in text for term in ("quality", "warranty", "defect", "recall")):
            score += 10
        if any(term in text for term in ("supply", "inventory", "tier", "sourcing", "logistics")):
            score += 7
        if any(term in text for term in ("dealer", "field service", "network", "service center")):
            score += 5
        return min(score, 30)

    def _urgency_score(self, profile: dict[str, Any]) -> int:
        signal_text = _safe_text(profile.get("financial_highlights")).lower()
        challenges = profile.get("business_challenges", [])
        challenges_count = len(challenges) if isinstance(challenges, list) else 0

        score = 6 + min(challenges_count * 4, 12)
        if any(term in signal_text for term in ("decline", "margin", "pressure", "volatility", "negative")):
            score += 7
        return min(score, 25)

    def _readiness_score(self, profile: dict[str, Any]) -> int:
        products = profile.get("products_services", [])
        locations = profile.get("manufacturing_locations", [])
        score = 6
        if isinstance(products, list):
            score += min(len(products), 5)
        if isinstance(locations, list):
            score += min(len(locations) * 2, 8)
        if _safe_text(profile.get("company_type")).lower() in {"oem", "tier-1 supplier"}:
            score += 3
        return min(score, 20)

    def _evidence_depth_score(self, profile: dict[str, Any]) -> tuple[int, int, float]:
        citations = profile.get("citations", [])
        total = len(citations) if isinstance(citations, list) else 0
        high_conf = 0
        confidences: list[float] = []
        if isinstance(citations, list):
            for item in citations:
                if not isinstance(item, dict):
                    continue
                conf = _to_float(item.get("confidence"), 0.0)
                confidences.append(conf)
                if conf >= self.config.min_confidence:
                    high_conf += 1

        avg_conf = round(sum(confidences) / len(confidences), 3) if confidences else 0.0

        score = min(high_conf * 3, 15)
        if score == 0 and total > 0:
            score = min(total, 5)
        return score, total, avg_conf

    def _solution_match_score(self, profile: dict[str, Any]) -> tuple[int, str]:
        company_type = _safe_text(profile.get("company_type"))
        text = " ".join(profile.get("business_challenges", [])) if isinstance(profile.get("business_challenges"), list) else ""
        text = text.lower()

        if company_type == "OEM":
            solution = "Warranty Analytics + Dealer & Field Service Intelligence"
            base = 8
        elif company_type == "Tier-1 Supplier":
            solution = "Supply-Chain Risk Prediction + Warranty Analytics"
            base = 7
        else:
            solution = "Supply-Chain Risk Prediction"
            base = 6

        if any(term in text for term in ("dealer", "service")) and "Dealer" in solution:
            base += 2
        if any(term in text for term in ("supply", "inventory", "tier")) and "Supply-Chain" in solution:
            base += 2
        return min(base, 10), solution

    def _gate_result(self, total_citations: int, avg_confidence: float) -> bool:
        return total_citations >= self.config.min_citations and avg_confidence >= self.config.min_confidence

    def _generate_explanations(
        self,
        profile: dict[str, Any],
        scorecard: dict[str, Any],
        solution: str,
    ) -> dict[str, str]:
        payload = {
            "company_name": profile.get("company_name"),
            "company_type": profile.get("company_type"),
            "scorecard": scorecard,
            "solution": solution,
            "overview": profile.get("company_overview"),
            "growth": profile.get("business_growth"),
            "challenges": profile.get("business_challenges"),
        }
        prompt = (
            "Return JSON with keys: why_selected, relevant_consulting_solution, "
            "expected_business_value, why_strong_potential_customer."
            f"\n\nPayload:\n{json.dumps(payload, ensure_ascii=False)}"
        )
        run = self.prioritizer_agent.run(prompt)
        text = _safe_text(getattr(run, "content", ""))
        try:
            parsed = json.loads(text)
        except Exception:  # noqa: BLE001
            parsed = {}

        return {
            "why_selected": _safe_text(parsed.get("why_selected")) or "Selected due to scorecard strength and evidence-backed consulting fit.",
            "relevant_consulting_solution": _safe_text(parsed.get("relevant_consulting_solution")) or solution,
            "expected_business_value": _safe_text(parsed.get("expected_business_value")) or "Expected value: improved operational KPIs, risk reduction, and margin resilience.",
            "why_strong_potential_customer": _safe_text(parsed.get("why_strong_potential_customer")) or "Strong potential due to demonstrated business need and consultancy-solution alignment.",
        }

    def prioritize(self, profiles: list[dict[str, Any]], top_n: int = 10) -> dict[str, Any]:
        ranked: list[dict[str, Any]] = []

        for profile in profiles:
            service_alignment = self._service_alignment_score(profile)
            urgency = self._urgency_score(profile)
            readiness = self._readiness_score(profile)
            evidence_depth, total_citations, avg_conf = self._evidence_depth_score(profile)
            solution_match, solution = self._solution_match_score(profile)

            total_score = service_alignment + urgency + readiness + evidence_depth + solution_match
            gate_pass = self._gate_result(total_citations=total_citations, avg_confidence=avg_conf)

            scorecard = {
                "service_alignment_0_to_30": service_alignment,
                "urgency_0_to_25": urgency,
                "execution_readiness_0_to_20": readiness,
                "evidence_depth_0_to_15": evidence_depth,
                "solution_match_0_to_10": solution_match,
                "composite_score_0_to_100": total_score,
                "total_citations": total_citations,
                "avg_citation_confidence": avg_conf,
                "gate_pass": gate_pass,
            }

            explanations = self._generate_explanations(profile=profile, scorecard=scorecard, solution=solution)

            ranked.append(
                {
                    "company_name": profile.get("company_name"),
                    "company_type": profile.get("company_type"),
                    "ticker": profile.get("ticker", ""),
                    "scoring": scorecard,
                    "most_relevant_solution": solution,
                    "why_selected": explanations["why_selected"],
                    "relevant_consulting_solution": explanations["relevant_consulting_solution"],
                    "expected_business_value": explanations["expected_business_value"],
                    "why_strong_potential_customer": explanations["why_strong_potential_customer"],
                    "citations": profile.get("citations", []),
                }
            )

        ranked.sort(
            key=lambda row: (
                bool(row["scoring"].get("gate_pass", False)),
                row["scoring"].get("composite_score_0_to_100", 0),
                row["scoring"].get("service_alignment_0_to_30", 0),
                row["scoring"].get("evidence_depth_0_to_15", 0),
            ),
            reverse=True,
        )

        top = ranked[:top_n]
        for idx, row in enumerate(top, start=1):
            row["rank"] = idx

        return {
            "generated_at": _now_iso(),
            "top_n": top_n,
            "total_candidates": len(ranked),
            "selected": top,
            "gate_summary": {
                "min_citations": self.config.min_citations,
                "min_confidence": self.config.min_confidence,
                "pass_count": sum(1 for r in ranked if r["scoring"].get("gate_pass")),
                "fail_count": sum(1 for r in ranked if not r["scoring"].get("gate_pass")),
            },
        }


This cell defines specific configuration parameters related to company data, including the ChromaDB collection name for company profiles and the path to the company signals JSON file.

In [49]:
company_collection="company_profiles_chunks"
company_signals=Path("/content/extraction_output/company_signals.json")

This cell defines the path to the market report JSON file, which contains aggregated market research data used as input for company research and prioritization.

In [50]:
market_report = Path("/content/extraction_output/market_research/report.json")

This cell orchestrates the final phase of company research, combining market insights with detailed company profiling and prioritization. It performs the following key steps:

1.  **Loads Market Report**: It loads the `market_research/report.json` file, which contains the aggregated market research data from previous steps.
2.  **Runs Company Ingestion**: It executes `run_company_ingestion` to collect financial signals and news snippets for a predefined list of companies, indexing this data into a ChromaDB collection (`company_profiles_chunks`).
3.  **Generates Candidate Pool**: It creates a list of candidate companies for profiling by combining entries from the predefined `COMPANY_CATALOG` and any organizations identified in the loaded market report.
4.  **Initializes Company Research Orchestrator**: An instance of `CompanyResearchOrchestrator` is created, configured with paths to the ChromaDB and other necessary files, and an OpenAI model for agent-based profiling.
5.  **Profiles Companies**: It runs the `research_orchestrator` to generate detailed profiles for each candidate company. These profiles are created by querying the ChromaDB for relevant market and company-specific information.
6.  **Initializes Company Prioritization Orchestrator**: An instance of `CompanyPrioritizationOrchestrator` is created, configured with an OpenAI model and thresholds for citation count and confidence.
7.  **Prioritizes Companies**: It uses the `prioritizer` to score and rank the profiled companies based on various criteria (e.g., service alignment, urgency, readiness, evidence depth). The top 10 companies are selected.
8.  **Saves Outputs**: The cell then saves the generated data to the `extraction_output/company_research` directory:
    *   `company_profiles.jsonl`: Individual company profiles in JSONL format.
    *   `company_profiles.json`: All company profiles aggregated into a single JSON file.
    *   `company_prioritization.json`: The full prioritization report, including scores and explanations for all ranked companies.
    *   `prioritized_top10.md`: A human-readable markdown report of the top 10 prioritized companies.
    *   `run_manifest.json`: A manifest detailing the parameters and artifacts of this company research run.
9.  **Prints Summary**: Finally, it prints a summary of the completed company research run, including the number of candidates and profiles, and the paths to the main output files.

This cell orchestrates the final phase of company research, combining market insights with detailed company profiling and prioritization. It performs the following key steps:

1.  **Loads Market Report**: It loads the `market_research/report.json` file, which contains the aggregated market research data from previous steps.
2.  **Runs Company Ingestion**: It executes `run_company_ingestion` to collect financial signals and news snippets for a predefined list of companies, indexing this data into a ChromaDB collection (`company_profiles_chunks`).
3.  **Generates Candidate Pool**: It creates a list of candidate companies for profiling by combining entries from the predefined `COMPANY_CATALOG` and any organizations identified in the loaded market report.
4.  **Initializes Company Research Orchestrator**: An instance of `CompanyResearchOrchestrator` is created, configured with paths to the ChromaDB and other necessary files, and an OpenAI model for agent-based profiling.
5.  **Profiles Companies**: It runs the `research_orchestrator` to generate detailed profiles for each candidate company. These profiles are created by querying the ChromaDB for relevant market and company-specific information.
6.  **Initializes Company Prioritization Orchestrator**: An instance of `CompanyPrioritizationOrchestrator` is created, configured with an OpenAI model and thresholds for citation count and confidence.
7.  **Prioritizes Companies**: It uses the `prioritizer` to score and rank the profiled companies based on various criteria (e.g., service alignment, urgency, readiness, evidence depth). The top 10 companies are selected.
8.  **Saves Outputs**: The cell then saves the generated data to the `extraction_output/company_research` directory:
    *   `company_profiles.jsonl`: Individual company profiles in JSONL format.
    *   `company_profiles.json`: All company profiles aggregated into a single JSON file.
    *   `company_prioritization.json`: The full prioritization report, including scores and explanations for all ranked companies.
    *   `prioritized_top10.md`: A human-readable markdown report of the top 10 prioritized companies.
    *   `run_manifest.json`: A manifest detailing the parameters and artifacts of this company research run.
9.  **Prints Summary**: Finally, it prints a summary of the completed company research run, including the number of candidates and profiles, and the paths to the main output files.

In [51]:
import argparse
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def _candidate_pool(market_report_content: dict[str, Any]) -> list[dict[str, str]]:
    pool: list[dict[str, str]] = []
    seen: set[str] = set()

    for item in COMPANY_CATALOG:
        name = _safe_text(item.get("name"))
        if not name:
            continue
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        pool.append({"name": name, "category": _safe_text(item.get("company_type"))})

    research_orgs = market_report_content.get("research", {}).get("organizations", [])
    for item in research_orgs if isinstance(research_orgs, list) else []:
        if not isinstance(item, dict):
            continue
        name = _safe_text(item.get("name"))
        category = _safe_text(item.get("category"))
        if not name:
            continue
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        pool.append({"name": name, "category": category or "Unknown"})

    return pool

def _render_markdown(prioritization: dict[str, Any], topic: str) -> str:
    lines: list[str] = []
    lines.append("# Top 10 Company Prioritization")
    lines.append("")
    lines.append(f"Topic: {topic}")
    lines.append("")

    selected = prioritization.get("selected", [])
    for row in selected:
        lines.append(f"## Rank {row.get('rank')}: {row.get('company_name')} ({row.get('company_type')})")
        lines.append("")
        lines.append(f"- Composite score: {row.get('scoring', {}).get('composite_score_0_to_100')}")
        lines.append(f"- Gate pass: {row.get('scoring', {}).get('gate_pass')}")
        lines.append(f"- Why selected: {row.get('why_selected')}")
        lines.append(f"- Relevant consulting solution: {row.get('relevant_consulting_solution')}")
        lines.append(f"- Expected business value: {row.get('expected_business_value')}")
        lines.append(f"- Why strong potential customer: {row.get('why_strong_potential_customer')}")
        lines.append("")

    return "\n".join(lines).strip() + "\n"


market_report_path = Path("/content/extraction_output/market_research/report.json")
loaded_market_report = _load_json(market_report_path)
ingestion_result = run_company_ingestion(
            phase1_company_dir=path,
            phase3_company_dir=path,
            phase4_persist_dir=persist_dir,
            collection_name=company_collection,
            per_company_news_limit=4,
        )

candidates = _candidate_pool(loaded_market_report)

research_orchestrator = CompanyResearchOrchestrator(
        CompanyResearchConfig(
            persist_dir=persist_dir,
            market_collection=market_collection,
            company_collection=company_collection,
            company_signals_file=company_signals,
            openai_model=openai_model,
        )
    )

profiles = research_orchestrator.run(candidates)

prioritizer = CompanyPrioritizationOrchestrator(
        PrioritizationConfig(openai_model=openai_model, min_citations=2, min_confidence=0.74)
    )
prioritization = prioritizer.prioritize(profiles=profiles, top_n=10)

out_dir = Path("/content/extraction_output/company_research")
out_dir.mkdir(parents=True, exist_ok=True)

profiles_jsonl = out_dir / "company_profiles.jsonl"
profiles_json = out_dir / "company_profiles.json"
top_json = out_dir / "company_prioritization.json"
top_md = out_dir / "prioritized_top10.md"
manifest = out_dir / "run_manifest.json"

with profiles_jsonl.open("w", encoding="utf-8") as handle:
        for row in profiles:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

with profiles_json.open("w", encoding="utf-8") as handle:
        json.dump({"generated_at": _now_iso(), "profiles": profiles}, handle, indent=2, ensure_ascii=False)

with top_json.open("w", encoding="utf-8") as handle:
        json.dump(prioritization, handle, indent=2, ensure_ascii=False)

top_md.write_text(
        _render_markdown(
            prioritization=prioritization,
            topic=_safe_text(loaded_market_report.get("topic", "Company Prioritization")),
        ),
        encoding="utf-8",
    )

run_manifest = {
        "generated_at": _now_iso(),
        "market_report": loaded_market_report,
        "refresh_company_data": "true",
        "ingestion": ingestion_result,
        "candidate_count": len(candidates),
        "profile_count": len(profiles),
        "top_n": 10,
        "artifacts": {
            "profiles_jsonl": str(profiles_jsonl),
            "profiles_json": str(profiles_json),
            "prioritization_json": str(top_json),
            "prioritization_markdown": str(top_md),
        },
    }
with manifest.open("w", encoding="utf-8") as handle:
        json.dump(run_manifest, handle, indent=2, ensure_ascii=False)

print("Company research run complete")
print(f"- Candidates: {len(candidates)}")
print(f"- Profiles: {len(profiles)}")
print(f"- Top N: 10")
print(f"- Output JSON: {top_json}")
print(f"- Output Markdown: {top_md}")
print(f"- Run manifest: {manifest}")

ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$500570.BO: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


Company research run complete
- Candidates: 16
- Profiles: 16
- Top N: 10
- Output JSON: /content/extraction_output/company_research/company_prioritization.json
- Output Markdown: /content/extraction_output/company_research/prioritized_top10.md
- Run manifest: /content/extraction_output/company_research/run_manifest.json


In [53]:
import shutil

shutil.make_archive(
    "Hackathon-OuputFiles",
    "zip",
    "/content/extraction_output/"
)

from google.colab import files

files.download("/content/Hackathon-OuputFiles.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>